# Semana 7 — Ontología y grafo de conocimiento · Glaciares (3 capas)
**Proyecto:** Enterprise Brain — Gestión del Conocimiento con Knowledge Graph + GraphRAG · Grupo G8

**Entrada:** `tripletas_glaciares_v22/` (Persona 3: hechos con documento, página y evidencia) y `corpus_glaciares/papers.csv` (Persona 1)
**Salida:** `grafo_glaciares/` → ontología OWL, grafo en Neo4j, consultas Cypher y SPARQL, vistas interactivas

| Capa | Tecnología | Para qué |
|---|---|---|
| 1. Grafo principal | **Neo4j AuraDB** (Cypher) | Consultas de varios saltos, demo visual y base del GraphRAG (semanas 9–10) |
| 2. Capa semántica | **OWL + RDF + SPARQL** (rdflib) | Ontología alineada a **GeoSPARQL · SOSA/SSN · PROV-O**; **deduce** relaciones y **valida** los datos con las reglas de la ontología |
| 3. Visual y respaldo | **Pyvis** + archivos | Vistas interactivas sin internet; CSV para el importador de Neo4j; las preguntas también corren sin Neo4j |

### Antes de ejecutar (una sola vez)
1. Crear la base gratuita en **console.neo4j.io** → *New Instance* → *AuraDB Free*. Descargar el `.txt` con las credenciales (solo se muestra una vez).
2. En Colab, 🔑 **Secrets**: `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` (con *Notebook access* activado).
3. *Entorno de ejecución → Ejecutar todas* (no necesita GPU). Tiempo aprox.: 5–10 min.

Sin las claves de Neo4j el notebook **igual funciona**: hace la ontología, el RDF/SPARQL y las vistas, y salta la parte de Neo4j.

## 0. Carpetas

In [ ]:
BASE_LOCAL = r"G:\Mi unidad\proyecto_gc"

import os, sys
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/proyecto_gc"
else:
    BASE = BASE_LOCAL
CARPETA_CORPUS = os.path.join(BASE, "corpus_glaciares")
CARPETA_TRIP   = os.path.join(BASE, "tripletas_glaciares_v22")
CARPETA_GRAFO  = os.path.join(BASE, "grafo_glaciares")
os.makedirs(CARPETA_GRAFO, exist_ok=True)
print("Entrada:", CARPETA_TRIP, "\nSalida: ", CARPETA_GRAFO)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Entrada: /content/drive/MyDrive/proyecto_gc/tripletas_glaciares_v22 
Salida:  /content/drive/MyDrive/proyecto_gc/grafo_glaciares


## 1. Claves de Neo4j (desde Secrets)

In [ ]:
def leer_secreto(*nombres):
    for nombre in nombres:
        try:
            from google.colab import userdata
            v = userdata.get(nombre)
            if v:
                print(f"{nombre}: cargada desde Secrets")
                return v
        except Exception:
            pass
        if os.environ.get(nombre):
            print(f"{nombre}: cargada desde variable de entorno")
            return os.environ[nombre]
    print(f"{nombres[0]}: no configurada")
    return None

NEO4J_URI = leer_secreto("NEO4J_URI")
NEO4J_USER = leer_secreto("NEO4J_USER", "NEO4J_USERNAME") or "neo4j"
NEO4J_PASSWORD = leer_secreto("NEO4J_PASSWORD")

NEO4J_URI: cargada desde Secrets
NEO4J_USER: cargada desde Secrets
NEO4J_PASSWORD: cargada desde Secrets


## 2. Configuración ✏️

In [ ]:
CONFIG = {
    "CONF_MIN": 0.5,             # confianza mínima de un hecho para entrar al grafo
    "USAR_NEO4J": True,          # si no hay claves, se salta automáticamente
    "VACIAR_NEO4J": True,        # borra el grafo antes de cargar (para no duplicar al volver a ejecutar)
    "LOTE": 1000,                # filas por envío a Neo4j
    "ENTIDAD_DEMO": "Palcacocha",
    "DEPARTAMENTO_DEMO": "Áncash",
}

## 3. Código
Solo se ejecuta.

In [ ]:
%pip install -q rdflib pyvis neo4j

# -*- coding: utf-8 -*-
"""
Semana 7 — Ontología y grafo de conocimiento del corpus Glaciares (3 capas)
Proyecto: Enterprise Brain — Gestión del Conocimiento (Knowledge Graph + GraphRAG) · Grupo G8

Entrada (Persona 3): tripletas_glaciares_v22/tripletas_consolidadas.csv (+ entidades.csv) y corpus_glaciares/papers.csv
Capa 1: grafo de propiedades en Neo4j (consultas Cypher)
Capa 2: ontología OWL (GeoSPARQL · SOSA/SSN · PROV-O) + grafo RDF con SPARQL (deducción y validación)
Capa 3: vistas interactivas (Pyvis) y archivos de respaldo (CSV para el importador de Neo4j, Turtle, consultas)
Cada relación conserva su procedencia: documento, página, frase de evidencia y confianza.
"""
import os, re, json, gzip, time, hashlib, unicodedata
from collections import defaultdict, Counter

import pandas as pd

VERSION = "1.0-grafo"


def sin_tildes(t):
    return "".join(c for c in unicodedata.normalize("NFD", t or "") if unicodedata.category(c) != "Mn")


def norm(t):
    return re.sub(r"\s+", " ", sin_tildes(str(t or "").lower())).strip()


def clave(tipo, nombre):
    return f"{tipo}:{re.sub(r'[^a-z0-9]+', '_', norm(nombre)).strip('_')}"


def _h(*partes):
    return hashlib.md5("|".join(str(p) for p in partes).encode("utf-8")).hexdigest()[:16]


def _num(x):
    try:
        v = float(x)
        return None if v != v else v
    except (TypeError, ValueError):
        return None


def _int(x):
    v = _num(x)
    return int(v) if v is not None else None


def _txt(x, n=None):
    if x is None or (isinstance(x, float) and x != x):
        return ""
    s = re.sub(r"\s+", " ", str(x)).strip()
    return s[:n] if n else s


# ======================================================================
# 1. Modelo del grafo
# ======================================================================
# relación entre entidades → tipo de relación en Neo4j
REL = {
    "ubicadaEn": "UBICADA_EN", "alimentadaPor": "ALIMENTADA_POR", "amenazaA": "AMENAZA_A", "ocurrioEn": "OCURRIO_EN",
    "afectoA": "AFECTO_A", "tieneObraSeguridad": "TIENE_OBRA", "monitoreadoCon": "MONITOREADO_CON",
    "intervieneEn": "INTERVIENE_EN", "tieneEcosistema": "TIENE_ECOSISTEMA", "presentaFenomeno": "PRESENTA_FENOMENO",
    "albergaEspecie": "ALBERGA_ESPECIE", "proveeServicio": "PROVEE_SERVICIO", "abasteceA": "ABASTECE_A",
    "drenaHacia": "DRENA_HACIA", "relacionadoCon": "RELACIONADO_CON", "inventarioDe": "INVENTARIO_DE",
    "mencionaA": "MENCIONA", "usaMetodo": "USA_METODO",
}
# si el objeto llegó como texto, a qué tipo de nodo se convierte
OBJ_POR_DEFECTO = {"tieneObraSeguridad": "ObraSeguridad", "monitoreadoCon": "MetodoMonitoreo"}
# años: se guardan como propiedad del nodo
ANIOS = {"ocurrioEnAnio": "anio", "formadaEnAnio": "anio_formacion", "anioInventario": "anio", "anioImagenes": "anio_imagenes"}
NIVELES = ["Muy alto", "Alto", "Medio", "Bajo"]
EN_RIESGO = ["Muy alto", "Alto"]
METODOS_BALANCE = ["Balance de masa", "Estacas de ablación"]
HIDRO = {"Cuenca", "Quebrada"}
CUENCAS_GRANDES = {norm(x) for x in ["Santa", "Marañón", "Maranon", "Rímac", "Mantaro", "Urubamba", "Vilcanota", "Inambari",
                                     "Apurímac", "Ucayali", "Amazonas", "Huallaga", "Pativilca", "Pastaza", "Madre de Dios",
                                     "Ocoña", "Camaná", "Majes", "Colca", "Chillón", "Lurín", "Cañete", "Pisco", "Tambo",
                                     "Titicaca", "Pacífico", "Atlántico"]}
DEPARTAMENTOS = ["Áncash", "Junín", "Cusco", "Puno", "Huánuco", "Apurímac", "Huancavelica", "Arequipa", "Ayacucho",
                 "Pasco", "Cajamarca", "Moquegua", "Tacna", "Lima", "La Libertad", "Piura", "Amazonas", "San Martín",
                 "Ica", "Lambayeque", "Tumbes", "Loreto", "Ucayali", "Madre de Dios"]
# frases que niegan un evento ("no registrando actividad…") → el evento no se carga
NEGACION = re.compile(r"\bno\s+(se\s+)?(ha\s+|han\s+)?(registr|present|ocurri|report|evidenci|produj)|\bsin\s+(registro|evento|antecedente)|"
                      r"\bningún\s+(evento|aluvi)", re.I)
NOMBRES_GENERICOS = {"bofedal", "bofedales", "humedal", "humedales", "laguna", "lagunas", "glaciar", "glaciares"}


def nivel_canon(t):
    n = norm(t)
    if re.search(r"muy\s+alt|critic", n):
        return "Muy alto"
    if re.search(r"\balt", n):
        return "Alto"
    if re.search(r"\bmedi|moderad", n):
        return "Medio"
    if re.search(r"\bbaj", n):
        return "Bajo"
    return _txt(t).capitalize()


def aplicar_rechazos(cons, rechazos, log=print):
    """Curaduría negativa: quita hechos que el equipo revisó y marcó como errores.
    rechazos: DataFrame con columnas sujeto, relacion, objeto, motivo ('*' = cualquiera). Se compara sin tildes ni mayúsculas."""
    if rechazos is None or not len(rechazos):
        return cons, cons.iloc[0:0]
    s_n, p_n, o_n = cons.subject.map(norm), cons.predicate.astype(str), cons.object.map(norm)
    st, ot = cons.subject_type.astype(str), cons.object_type.astype(str)
    quitar = pd.Series(False, index=cons.index)
    rech = rechazos.fillna("*")
    if "tipo_sujeto" not in rech.columns:
        rech["tipo_sujeto"] = "*"
    for r in rech.itertuples():
        m = (s_n == norm(r.sujeto)) if str(r.sujeto).strip() != "*" else pd.Series(True, index=cons.index)
        tipo_ok = str(r.tipo_sujeto).strip()
        if tipo_ok not in ("*", ""):
            m &= st == tipo_ok
        if str(r.relacion).strip() != "*":
            m &= p_n == str(r.relacion).strip()
        if str(r.objeto).strip() != "*":
            m &= o_n == norm(r.objeto)
        quitar |= m
        # si se rechaza un evento entero, también se quitan los hechos donde aparece como objeto
        if str(r.relacion).strip() == "*" and str(r.objeto).strip() == "*" and str(r.sujeto).strip() != "*":
            quitar |= (o_n == norm(r.sujeto)) & ((ot == tipo_ok) if tipo_ok not in ("*", "") else True)
    log(f"Curaduría: {int(quitar.sum())} hechos rechazados por el equipo")
    return cons[~quitar], cons[quitar]


def preparar_grafo(cons, entidades=None, papers=None, conf_min=0.5, log=print):
    """De tripletas consolidadas a nodos y relaciones (con procedencia en cada relación).
    Devuelve dict con: nodos (DataFrame), rels (DataFrame), resumen (dict)."""
    c = cons.copy()
    for col in ("subject_id", "object_id", "object", "subject", "predicate", "subject_type", "object_type",
                "paper_id", "evidencia", "metodos", "verb", "documentos", "year"):
        if col in c.columns:
            c[col] = c[col].map(lambda x: _txt(x))
    c = c[(c.confianza.astype(float) >= conf_min) | c.predicate.isin(["mencionaA", "usaMetodo"])]
    # eventos negados
    ev = c[c.subject_type == "Evento"]
    negados = set(ev[ev.evidencia.map(lambda e: bool(NEGACION.search(e))).astype(bool)].subject_id)
    c = c[~c.subject_id.isin(negados) & ~c.object_id.isin(negados)]

    nodos = {}

    def nodo(nid, tipo, nombre, **props):
        if nid not in nodos:
            nodos[nid] = {"id": nid, "tipo": tipo, "nombre": _txt(nombre), "k": norm(nombre)}
        for k_, v in props.items():
            if v is not None and v != "" and k_ not in nodos[nid]:
                nodos[nid][k_] = v
        return nid

    # documentos (Persona 1)
    titulos, fuentes, anios_doc = {}, {}, {}
    if papers is not None:
        for r in papers.itertuples():
            pid = str(r.paper_id)
            titulos[pid], fuentes[pid] = _txt(getattr(r, "title", "")), _txt(getattr(r, "fuente", ""))
            anios_doc[pid] = _int(getattr(r, "year", None))
            nodo(pid, "Documento", titulos[pid] or pid, titulo=titulos[pid], fuente=fuentes[pid], anio=anios_doc[pid],
                 eje=_txt(getattr(r, "eje_principal", "")))
    n_docs_ent = dict(zip(entidades.entidad_id, entidades.n_docs)) if entidades is not None and len(entidades) else {}

    rels = []

    def rel(s, o, tipo, pred, r=None, **extra):
        p = {"pred": pred}
        if r is not None:
            p.update(documento=r.paper_id, pagina=_int(r.page_start), evidencia=_txt(r.evidencia, 400),
                     confianza=round(float(r.confianza), 2), n_docs=_int(r.n_docs) or 1, metodos=r.metodos,
                     documentos=r.documentos, anio=_int(r.year), verbo=r.verb)
        p.update({k_: v for k_, v in extra.items()})
        p = {k_: v for k_, v in p.items() if v is not None and v != ""}
        rels.append({"s": s, "o": o, "tipo": tipo, "fid": _h(s, tipo, o, p.get("documento", ""), p.get("obra", "")), **p})

    for r in c.itertuples():
        pred, st = r.predicate, r.subject_type
        if pred in ("mencionaA", "usaMetodo"):
            if r.subject_id not in nodos:
                nodo(r.subject_id, "Documento", r.subject_id)
            nodo(r.object_id, r.object_type, r.object, n_docs=n_docs_ent.get(r.object_id))
            rel(r.subject_id, r.object_id, REL[pred], pred, n=_int(r.value) or 1)
            continue
        s = nodo(r.subject_id, st, r.subject, n_docs=n_docs_ent.get(r.subject_id))
        if pred in REL and (r.object_type != "Literal" or pred in OBJ_POR_DEFECTO):
            ot = r.object_type if r.object_type != "Literal" else OBJ_POR_DEFECTO[pred]
            oid = r.object_id or clave(ot, r.object)
            nodo(oid, ot, r.object, n_docs=n_docs_ent.get(oid))
            rel(s, oid, REL[pred], pred, r)
        elif pred == "tieneNivelRiesgo":
            nv = nivel_canon(r.object)
            rel(s, nodo(clave("NivelRiesgo", nv), "NivelRiesgo", nv, orden=NIVELES.index(nv) if nv in NIVELES else 9),
                "TIENE_NIVEL_RIESGO", pred, r)
        elif pred == "estadoObra":
            obra, _, estado = r.object.partition(":")
            estado = (estado or obra).strip()
            rel(s, nodo(clave("EstadoObra", estado), "EstadoObra", estado), "ESTADO_OBRA", pred, r, obra=obra.strip())
        elif pred in ANIOS:
            a = _int(r.value) or _int(r.object[:4])
            if a:
                nodos[s].setdefault(ANIOS[pred], a)
        else:   # medición (sosa:Observation): volumen, área, altitud, superficie, conteos del inventario, víctimas…
            mid = "Medicion:" + _h(s, pred, r.object)
            nodo(mid, "Medicion", f"{pred} {r.object}", propiedad=pred, texto=r.object, valor=_num(r.value_std),
                 unidad=_txt(r.unit_std) or _txt(r.unit), anio=_int(r.year), documento=r.paper_id,
                 pagina=_int(r.page_start), evidencia=_txt(r.evidencia, 400), confianza=round(float(r.confianza), 2))
            rel(s, mid, "TIENE_MEDICION", pred, r)

    # documento → institución que lo publicó
    for pid, f in fuentes.items():
        if f:
            iid = clave("Institucion", f)
            nodo(iid, "Institucion", f)
            rel(pid, iid, "PUBLICADO_POR", "publicadoPor")

    N = pd.DataFrame(list(nodos.values()))
    R = pd.DataFrame(rels).drop_duplicates("fid")
    _derivar(N, R)
    resumen = {"nodos": len(N), "relaciones": len(R), "eventos_negados_quitados": len(negados),
               "por_tipo": N.tipo.value_counts().to_dict(), "por_relacion": R.tipo.value_counts().to_dict()}
    log(f"Nodos: {len(N):,} · relaciones: {len(R):,} · eventos negados quitados: {len(negados)}")
    return {"nodos": N, "rels": R, "resumen": resumen}


def _derivar(N, R):
    """Propiedades derivadas que simplifican las consultas (y se explican en el informe):
    departamentos (explícito > deducido), nivel de riesgo máximo, subcuencas pequeñas, nombres dudosos."""
    tipo = dict(zip(N.id, N.tipo))
    nombre = dict(zip(N.id, N.nombre))
    ub = R[R.tipo == "UBICADA_EN"]
    padres = ub.groupby("s").o.agg(set).to_dict()
    # 1) departamentos: si el texto lo dice, manda el de más respaldo; si no, se sube por la jerarquía
    dep = ub[ub.o.map(lambda x: tipo.get(x) == "Departamento").astype(bool)]
    puntaje = dep.assign(p=dep.confianza.fillna(0.5).astype(float) * dep.n_docs.fillna(1).astype(float)).groupby(["s", "o"]).p.sum()

    def ancestros(i):
        vistos, front = set(), {i}
        for _ in range(4):
            front = {p for x in front for p in padres.get(x, ())} - vistos - {i}
            if not front:
                break
            vistos |= front
        return vistos

    deps = {}
    con_dep = set(puntaje.index.get_level_values(0)) if len(puntaje) else set()
    for i, t in tipo.items():
        if t not in ("Laguna", "Glaciar", "Nevado", "Bofedal", "CentroPoblado", "Cuenca", "Quebrada", "Provincia", "Distrito"):
            continue
        if i in con_dep:
            s = puntaje.loc[i]
            ds = {o for o, v in s.items() if v >= 0.5 * s.max()}
        else:
            # cada lugar "padre" vota por los departamentos a los que llega; ganan los más votados
            # (así una microcuenca homónima en otra región no arrastra a la laguna a ese departamento)
            votos = Counter()
            for p_ in padres.get(i, ()):
                destino = {p_} if tipo.get(p_) == "Departamento" else {a for a in ancestros(p_) if tipo.get(a) == "Departamento"}
                votos.update(destino)
            ds = {d for d, v in votos.items() if v == max(votos.values())} if votos else set()
        if ds:
            deps[i] = sorted(nombre[d] for d in ds)
    N["departamentos"] = N.id.map(lambda i: deps.get(i, []))
    # 2) nivel de riesgo máximo citado (una laguna "en riesgo" si algún documento la cita Alto o Muy alto)
    nr = R[R.tipo == "TIENE_NIVEL_RIESGO"].assign(niv=lambda d: d.o.map(nombre))
    orden = nr.assign(i=nr.niv.map(lambda x: NIVELES.index(x) if x in NIVELES else 9)).groupby("s").i.min()
    N["nivel_max"] = N.id.map(lambda i: NIVELES[int(orden[i])] if i in orden.index and orden[i] < 9 else None)
    N["en_riesgo"] = N.nivel_max.isin(EN_RIESGO)
    # 3) subcuencas: cuántos poblados tienen; las grandes (río Santa, Marañón…) no sirven para "aguas abajo"
    pob = ub[ub.s.map(lambda x: tipo.get(x) == "CentroPoblado").astype(bool) & ub.o.map(lambda x: tipo.get(x) in HIDRO).astype(bool)]
    npob = pob.groupby("o").s.nunique()
    N["hidro"] = N.tipo.isin(HIDRO)
    N["n_poblados"] = N.id.map(lambda i: int(npob.get(i, 0)))
    N["grande"] = N.hidro & ((N.n_poblados > 12) | N.k.isin(CUENCAS_GRANDES))
    # 4) nombres dudosos: un "poblado" que en realidad es nevado/laguna/cordillera/departamento, o bofedales genéricos
    n_rel = pd.concat([R.s, R.o]).value_counts()
    peso = N.assign(n=N.id.map(lambda i: int(n_rel.get(i, 0))))
    otros = peso[peso.tipo.isin(["Nevado", "Glaciar", "Laguna", "Cordillera", "Departamento"])].groupby("k").n.sum()
    pob_n = peso[peso.tipo == "CentroPoblado"].groupby("k").n.sum()
    deps_k = {norm(d) for d in DEPARTAMENTOS}
    otros_b = set(peso[peso.tipo.isin(["Nevado", "Glaciar", "Laguna", "Cordillera"])].k)   # "bofedal Pitec" junto al poblado Pitec es normal
    N["dudoso"] = [(t == "CentroPoblado" and (k in deps_k or pob_n.get(k, 0) <= otros.get(k, -1))) or
                   (t == "Bofedal" and (k in otros_b or k in NOMBRES_GENERICOS or len(k) < 3))
                   for t, k in zip(N.tipo, N.k)]
    return N


# ======================================================================
# 2. Ontología OWL (alineada a GeoSPARQL, SOSA/SSN y PROV-O)
# ======================================================================
BASE_ONT = "https://w3id.org/g8-glaciares/ontologia#"
BASE_REC = "https://w3id.org/g8-glaciares/recurso/"

# clase → (superclases propias, superclases de estándares, comentario)
CLASES = {
    "RasgoGeografico": ([], ["geo:Feature"], "Elemento del paisaje de montaña con ubicación (GeoSPARQL Feature)."),
    "MasaDeHielo": (["RasgoGeografico"], [], "Hielo glaciar: glaciares y nevados."),
    "Glaciar": (["MasaDeHielo"], [], "Masa de hielo permanente; se inventaría por cordillera."),
    "Nevado": (["MasaDeHielo"], [], "Montaña con cobertura glaciar."),
    "Laguna": (["RasgoGeografico"], [], "Laguna de origen glaciar (periglaciar, proglaciar o supraglaciar)."),
    "Bofedal": (["RasgoGeografico", "Ecosistema"], [], "Humedal altoandino con nombre propio."),
    "Cordillera": (["RasgoGeografico"], [], "Cordillera glaciar del Perú (20 según el INAIGEM)."),
    "UnidadHidrografica": (["RasgoGeografico"], [], "Cuenca, subcuenca, microcuenca o quebrada."),
    "Cuenca": (["UnidadHidrografica"], [], "Cuenca, subcuenca o microcuenca."),
    "Quebrada": (["UnidadHidrografica"], [], "Quebrada o valle glaciar."),
    "Rio": (["RasgoGeografico"], [], "Río."),
    "UnidadTerritorial": ([], ["geo:Feature"], "Unidad político-administrativa o asentamiento."),
    "Departamento": (["UnidadTerritorial"], [], "Departamento (región) del Perú."),
    "Provincia": (["UnidadTerritorial"], [], "Provincia."),
    "Distrito": (["UnidadTerritorial"], [], "Distrito."),
    "CentroPoblado": (["UnidadTerritorial"], [], "Ciudad, centro poblado o comunidad expuesta."),
    "Evento": ([], ["prov:Activity"], "Aluvión, avalancha, desborde (GLOF) u otro evento con fecha."),
    "ObraSeguridad": ([], [], "Dique, túnel, sifón, canal de desagüe o sistema de alerta temprana."),
    "NivelRiesgo": ([], [], "Categoría de riesgo de desborde: Muy alto, Alto, Medio, Bajo."),
    "EstadoObra": ([], [], "Estado de una obra de seguridad (operativa, deteriorada, colapsada…)."),
    "Ecosistema": ([], [], "Ecosistema altoandino (bofedal, pajonal, bosque de Polylepis…)."),
    "Especie": ([], [], "Especie de flora o fauna."),
    "ServicioEcosistemico": ([], [], "Beneficio que brinda un ecosistema (regulación hídrica, agua para riego…)."),
    "Fenomeno": ([], [], "Proceso o problema ambiental (retroceso glaciar, drenaje ácido de roca…)."),
    "MetodoMonitoreo": ([], ["sosa:Procedure"], "Método o técnica de monitoreo (batimetría, drones, InSAR…)."),
    "Medicion": ([], ["sosa:Observation"], "Valor medido (volumen, área, altitud, superficie, conteos) con unidad, año y fuente."),
    "Inventario": ([], ["prov:Entity"], "Inventario de glaciares y lagunas de una cordillera en un año."),
    "Documento": ([], ["prov:Entity"], "Documento técnico del corpus (INAIGEM, ANA, SENAMHI)."),
    "Institucion": ([], ["prov:Organization"], "Institución que publica, monitorea o ejecuta obras."),
    "Hecho": (["prov:Entity"], [], "Afirmación extraída de un documento (sujeto, relación, objeto) con su evidencia."),
}
# relaciones entre entidades: (dominio, rango, superpropiedad estándar, transitiva, comentario)
PROPIEDADES = {
    "ubicadaEn": ("geo:Feature", "geo:Feature", "geo:sfWithin", True, "Está dentro de (laguna → subcuenca → provincia → departamento)."),
    "alimentadaPor": ("Laguna", "MasaDeHielo", None, False, "La laguna recibe agua de deshielo de un glaciar o nevado."),
    "amenazaA": ("RasgoGeografico", "UnidadTerritorial", None, False, "Un desborde podría afectar a este poblado (aguas abajo)."),
    "ocurrioEn": ("Evento", "RasgoGeografico", "prov:atLocation", False, "Lugar de origen del evento."),
    "afectoA": ("Evento", "UnidadTerritorial", None, False, "Poblado o zona afectada por el evento."),
    "tieneObraSeguridad": ("Laguna", "ObraSeguridad", None, False, "Obra de seguridad construida en la laguna."),
    "tieneNivelRiesgo": ("Laguna", "NivelRiesgo", None, False, "Nivel de riesgo de desborde citado en un documento."),
    "estadoObra": ("Laguna", "EstadoObra", None, False, "Estado de una obra de la laguna (la obra va como dato de la relación)."),
    "monitoreadoCon": ("RasgoGeografico", "MetodoMonitoreo", None, False, "Método con el que se estudió o monitorea."),
    "intervieneEn": ("Institucion", "RasgoGeografico", None, False, "Institución que monitorea, evalúa o ejecuta obras."),
    "tieneEcosistema": ("geo:Feature", "Ecosistema", None, False, "Ecosistema presente en el lugar."),
    "presentaFenomeno": ("geo:Feature", "Fenomeno", None, False, "Fenómeno que ocurre en el lugar."),
    "albergaEspecie": ("geo:Feature", "Especie", None, False, "Especie presente."),
    "proveeServicio": ("geo:Feature", "ServicioEcosistemico", None, False, "Servicio ecosistémico que brinda."),
    "abasteceA": ("RasgoGeografico", "UnidadTerritorial", None, False, "Abastece de agua a."),
    "drenaHacia": ("RasgoGeografico", "RasgoGeografico", None, False, "Desagua hacia."),
    "relacionadoCon": (None, None, None, False, "Relación abierta encontrada por análisis gramatical (el verbo va como dato)."),
    "inventarioDe": ("Inventario", "Cordillera", None, False, "Cordillera que describe el inventario."),
    "tieneMedicion": ("geo:Feature", "Medicion", None, False, "Medición de una propiedad (inversa de sosa:hasFeatureOfInterest)."),
    "menciona": ("Documento", None, None, False, "El documento habla de la entidad."),
    "usaMetodo": ("Documento", "MetodoMonitoreo", None, False, "El documento aplica o describe el método."),
    "publicadoPor": ("Documento", "Institucion", "prov:wasAttributedTo", False, "Institución autora del documento."),
}
DATOS = {   # propiedades de datos: (dominio, tipo xsd, superpropiedad estándar)
    "anio": (None, "integer", None), "anioImagenes": ("Inventario", "integer", None),
    "valor": ("Medicion", "double", "sosa:hasSimpleResult"), "unidad": ("Medicion", "string", None),
    "propiedadMedida": ("Medicion", "string", None), "pagina": (None, "integer", None),
    "evidencia": (None, "string", None), "confianza": (None, "double", None), "fuente": ("Documento", "string", None),
}
PRED_REL_INV = {v: k for k, v in REL.items()}
PRED_REL_INV.update({"TIENE_NIVEL_RIESGO": "tieneNivelRiesgo", "ESTADO_OBRA": "estadoObra", "TIENE_MEDICION": "tieneMedicion",
                     "MENCIONA": "menciona", "PUBLICADO_POR": "publicadoPor", "TIENE_OBRA": "tieneObraSeguridad"})


def _ns():
    from rdflib import Namespace
    return {"eb": Namespace(BASE_ONT), "ent": Namespace(BASE_REC),
            "geo": Namespace("http://www.opengis.net/ont/geosparql#"), "sosa": Namespace("http://www.w3.org/ns/sosa/"),
            "prov": Namespace("http://www.w3.org/ns/prov#"), "skos": Namespace("http://www.w3.org/2004/02/skos/core#"),
            "dcterms": Namespace("http://purl.org/dc/terms/")}


def _iri(ns, q):
    pref, _, local = q.partition(":")
    return ns[pref][local] if local else ns["eb"][q]


def construir_ontologia():
    """Devuelve un rdflib.Graph con la ontología (clases, jerarquía, propiedades con dominio y rango, alineaciones)."""
    from rdflib import Graph, Literal, URIRef, BNode
    from rdflib.namespace import RDF, RDFS, OWL, XSD
    from rdflib.collection import Collection
    ns = _ns()
    g = Graph()
    for k, v in ns.items():
        g.bind(k, v)
    o = URIRef(BASE_ONT.rstrip("#"))
    g.add((o, RDF.type, OWL.Ontology))
    g.add((o, RDFS.label, Literal("Ontología Enterprise Brain Glaciares (G8)", lang="es")))
    g.add((o, ns["dcterms"].description, Literal(
        "Riesgo glaciar en el Perú: glaciares, lagunas, poblados expuestos, eventos, obras, monitoreo y procedencia de cada hecho. "
        "Alineada a GeoSPARQL (lugares), SOSA/SSN (mediciones y métodos) y PROV-O (procedencia).", lang="es")))
    for imp in ("http://www.opengis.net/ont/geosparql", "http://www.w3.org/ns/sosa/", "http://www.w3.org/ns/prov-o"):
        g.add((o, RDFS.seeAlso, URIRef(imp)))
    for ext in ("geo:Feature", "sosa:Procedure", "sosa:Observation", "prov:Entity", "prov:Activity", "prov:Organization"):
        g.add((_iri(ns, ext), RDF.type, OWL.Class))
    for cl, (sup, ext, com) in CLASES.items():
        u = ns["eb"][cl]
        g.add((u, RDF.type, OWL.Class))
        g.add((u, RDFS.label, Literal(re.sub(r"(?<!^)(?=[A-Z])", " ", cl).lower().capitalize(), lang="es")))
        g.add((u, RDFS.comment, Literal(com, lang="es")))
        for s in sup:
            g.add((u, RDFS.subClassOf, _iri(ns, s)))
        for e in ext:
            g.add((u, RDFS.subClassOf, _iri(ns, e)))
    # correspondencias con ENVO (ontología de ambientes) como referencias cercanas
    g.add((ns["eb"]["Laguna"], ns["skos"].closeMatch, URIRef("http://purl.obolibrary.org/obo/ENVO_00000020")))
    g.add((ns["eb"]["Glaciar"], ns["skos"].closeMatch, URIRef("http://purl.obolibrary.org/obo/ENVO_00000133")))
    # clases disjuntas básicas (sentido común: una laguna no es un poblado)
    for a, b in (("RasgoGeografico", "UnidadTerritorial"), ("Laguna", "MasaDeHielo"), ("Documento", "RasgoGeografico")):
        g.add((ns["eb"][a], OWL.disjointWith, ns["eb"][b]))
    for p, (dom, rng, sup, trans, com) in PROPIEDADES.items():
        u = ns["eb"][p]
        g.add((u, RDF.type, OWL.ObjectProperty))
        g.add((u, RDFS.label, Literal(re.sub(r"(?<!^)(?=[A-Z])", " ", p).lower(), lang="es")))
        g.add((u, RDFS.comment, Literal(com, lang="es")))
        if dom:
            g.add((u, RDFS.domain, _iri(ns, dom)))
        if rng:
            g.add((u, RDFS.range, _iri(ns, rng)))
        if sup:
            g.add((_iri(ns, sup), RDF.type, OWL.ObjectProperty))
            g.add((u, RDFS.subPropertyOf, _iri(ns, sup)))
        if trans:
            g.add((u, RDF.type, OWL.TransitiveProperty))
    g.add((ns["eb"]["tieneMedicion"], OWL.inverseOf, ns["sosa"]["hasFeatureOfInterest"]))
    g.add((ns["sosa"]["hasFeatureOfInterest"], RDF.type, OWL.ObjectProperty))
    for p, (dom, xsd, sup) in DATOS.items():
        u = ns["eb"][p]
        g.add((u, RDF.type, OWL.DatatypeProperty))
        if dom:
            g.add((u, RDFS.domain, _iri(ns, dom)))
        g.add((u, RDFS.range, XSD[xsd]))
        if sup:
            g.add((_iri(ns, sup), RDF.type, OWL.DatatypeProperty))
            g.add((u, RDFS.subPropertyOf, _iri(ns, sup)))
    # individuos fijos: niveles de riesgo
    for i, nv in enumerate(NIVELES):
        u = ns["eb"]["nivel_" + clave("x", nv).split(":")[1]]
        g.add((u, RDF.type, ns["eb"]["NivelRiesgo"]))
        g.add((u, RDFS.label, Literal(nv, lang="es")))
    return g


def resumen_ontologia(g):
    from rdflib.namespace import RDF, OWL
    eb = BASE_ONT
    clases = sorted(str(s)[len(eb):] for s in g.subjects(RDF.type, OWL.Class) if str(s).startswith(eb))
    obj = sorted(str(s)[len(eb):] for s in g.subjects(RDF.type, OWL.ObjectProperty) if str(s).startswith(eb))
    dat = sorted(str(s)[len(eb):] for s in g.subjects(RDF.type, OWL.DatatypeProperty) if str(s).startswith(eb))
    return {"clases": clases, "propiedades_objeto": obj, "propiedades_datos": dat, "triples": len(g)}


# ======================================================================
# 3. Capa 1 — Neo4j (grafo de propiedades) y consultas Cypher
# ======================================================================
def _limpio(d):
    out = {}
    for k, v in d.items():
        if v is None or (isinstance(v, float) and v != v):
            continue
        if hasattr(v, "item") and not isinstance(v, (list, tuple)):
            v = v.item()          # numpy → python
        if isinstance(v, float) and k in ("anio", "anio_imagenes", "anio_formacion", "pagina", "n_docs", "n", "n_poblados", "orden"):
            v = int(v)
        if isinstance(v, (list, tuple)):
            if not len(v):
                continue          # Neo4j no guarda listas vacías con tipo: se omite la propiedad
            v = [str(x) for x in v]
        out[k] = v
    return out


def filas_nodos(N):
    for t, g in N.groupby("tipo"):
        yield t, [{"id": r["id"], "props": _limpio({k: v for k, v in r.items() if k != "tipo" or True})} for r in g.to_dict("records")]


def filas_rels(R):
    cols_fijas = {"s", "o", "tipo"}
    for t, g in R.groupby("tipo"):
        yield t, [{"s": r["s"], "o": r["o"], "fid": r["fid"],
                   "props": _limpio({k: v for k, v in r.items() if k not in cols_fijas})} for r in g.to_dict("records")]


class Neo4jGrafo:
    """Conexión a Neo4j (AuraDB Free u otro) con carga por lotes y consultas que devuelven DataFrames."""

    def __init__(self, uri, usuario, clave_, base=None):
        from neo4j import GraphDatabase
        self.driver = GraphDatabase.driver(uri, auth=(usuario, clave_))
        self.driver.verify_connectivity()
        self.base = base

    def cerrar(self):
        self.driver.close()

    def q(self, cypher, **params):
        """Ejecuta una consulta y devuelve un DataFrame."""
        # base=None → la base "de casa" del usuario (en Aura nuevas no se llama 'neo4j', sino como la instancia)
        extra = {"database_": self.base} if self.base else {}
        recs, _, keys = self.driver.execute_query(cypher, params, **extra)
        return pd.DataFrame([r.data() for r in recs], columns=keys)

    def vaciar(self, log=print):
        total = 0
        while True:
            n = self.q("MATCH (n) WITH n LIMIT 5000 DETACH DELETE n RETURN count(*) AS n").n.iloc[0]
            total += int(n)
            if not n:
                break
        log(f"Grafo vaciado ({total:,} nodos borrados)")

    def esquema(self):
        for c in ["CREATE CONSTRAINT entidad_id IF NOT EXISTS FOR (n:Entidad) REQUIRE n.id IS UNIQUE",
                  "CREATE INDEX entidad_k IF NOT EXISTS FOR (n:Entidad) ON (n.k)",
                  "CREATE INDEX entidad_tipo IF NOT EXISTS FOR (n:Entidad) ON (n.tipo)",
                  "CREATE FULLTEXT INDEX nombres IF NOT EXISTS FOR (n:Entidad) ON EACH [n.nombre]"]:
            self.q(c)

    def cargar(self, N, R, lote=1000, log=print):
        t0 = time.time()
        for tipo, filas in filas_nodos(N):
            assert re.fullmatch(r"[A-Za-z]+", tipo), tipo
            for i in range(0, len(filas), lote):
                self.q(f"UNWIND $filas AS f MERGE (n:Entidad {{id: f.id}}) SET n += f.props SET n:{tipo}", filas=filas[i:i + lote])
        log(f"   nodos cargados · {time.time() - t0:.0f} s")
        for tipo, filas in filas_rels(R):
            assert re.fullmatch(r"[A-Z_]+", tipo), tipo
            for i in range(0, len(filas), lote):
                self.q(f"UNWIND $filas AS f MATCH (a:Entidad {{id: f.s}}) MATCH (b:Entidad {{id: f.o}}) "
                       f"MERGE (a)-[x:{tipo} {{fid: f.fid}}]->(b) SET x += f.props", filas=filas[i:i + lote])
        log(f"   relaciones cargadas · {time.time() - t0:.0f} s")

    def conteos(self):
        n = self.q("MATCH (n:Entidad) RETURN n.tipo AS tipo, count(*) AS nodos ORDER BY nodos DESC")
        r = self.q("MATCH ()-[r]->() RETURN type(r) AS relacion, count(*) AS relaciones ORDER BY relaciones DESC")
        return n, r


P_RIESGO = {"niveles": EN_RIESGO}
CONSULTAS = {
    "P1": dict(
        titulo="¿Cuántas lagunas de origen glaciar se formaron en la Cordillera Blanca según el inventario de 2023?",
        params={"cordillera": "blanca", "anio": 2023},
        cypher="""
MATCH (i:Inventario)-[:INVENTARIO_DE]->(c:Cordillera {k: $cordillera})
WHERE i.anio = $anio
MATCH (i)-[:TIENE_MEDICION]->(m:Medicion)
WHERE m.propiedad IN ['registraLagunas', 'registraLagunasNuevas', 'registraLagunasProglaciares', 'registraGlaciares', 'registraSuperficieGlaciar']
RETURN i.nombre AS inventario, m.propiedad AS dato, m.texto AS valor,
       m.documento AS documento, m.pagina AS pagina, m.evidencia AS evidencia
ORDER BY dato"""),
    "P2": dict(
        titulo="¿Qué lagunas glaciares de Áncash tienen riesgo muy alto de desborde?",
        params={"departamento": "Áncash", "nivel": "Muy alto"},
        cypher="""
MATCH (l:Laguna)-[r:TIENE_NIVEL_RIESGO]->(n:NivelRiesgo)
WHERE n.nombre = $nivel AND $departamento IN l.departamentos
RETURN l.nombre AS laguna, n.nombre AS nivel, r.n_docs AS n_docs,
       r.documento AS documento, r.pagina AS pagina, r.evidencia AS evidencia
ORDER BY n_docs DESC, laguna"""),
    "P3": dict(
        titulo="¿Qué centros poblados están aguas abajo de lagunas con riesgo alto y qué estudios los evaluaron?",
        params=P_RIESGO,
        cypher="""
MATCH (l:Laguna)-[r:AMENAZA_A]->(p:CentroPoblado)
WHERE l.nivel_max IN $niveles AND p.dudoso = false
OPTIONAL MATCH (d:Documento)-[:MENCIONA]->(l), (d)-[:MENCIONA]->(p)
RETURN l.nombre AS laguna, l.nivel_max AS nivel, p.nombre AS poblado, 'el texto lo dice' AS via, '' AS subcuenca,
       collect(DISTINCT d.id) AS estudios, r.documento AS documento, r.pagina AS pagina, r.evidencia AS evidencia
UNION
MATCH (l:Laguna)-[:UBICADA_EN]->(c)<-[r:UBICADA_EN]-(p:CentroPoblado)
WHERE l.nivel_max IN $niveles AND c.hidro = true AND c.grande = false AND p.dudoso = false
OPTIONAL MATCH (d:Documento)-[:MENCIONA]->(l), (d)-[:MENCIONA]->(p)
RETURN l.nombre AS laguna, l.nivel_max AS nivel, p.nombre AS poblado, 'misma subcuenca' AS via, c.nombre AS subcuenca,
       collect(DISTINCT d.id) AS estudios, r.documento AS documento, r.pagina AS pagina, r.evidencia AS evidencia"""),
    "P4": dict(
        titulo="¿Qué lagunas tuvieron eventos de desborde, de qué glaciar provienen y qué poblados afectaron?",
        params={},
        cypher="""
MATCH (e:Evento)-[r:OCURRIO_EN]->(l:Laguna)
OPTIONAL MATCH (l)-[:ALIMENTADA_POR]->(g)
OPTIONAL MATCH (e)-[:AFECTO_A]->(p)
OPTIONAL MATCH (e)-[:TIENE_MEDICION]->(v:Medicion)
WHERE v.propiedad = 'causoVictimas'
RETURN e.nombre AS evento, e.anio AS anio, l.nombre AS laguna,
       collect(DISTINCT g.nombre) AS glaciar_origen, collect(DISTINCT p.nombre) AS poblados_afectados,
       collect(DISTINCT v.texto) AS victimas, r.documento AS documento, r.pagina AS pagina, r.evidencia AS evidencia
ORDER BY anio"""),
    "P5": dict(
        titulo="¿Qué glaciares con retroceso medido (balance de masa) alimentan lagunas que hoy están en riesgo?",
        params={**P_RIESGO, "balance": METODOS_BALANCE},
        cypher="""
MATCH (l:Laguna)-[ra:ALIMENTADA_POR]->(g)
WHERE l.nivel_max IN $niveles AND g.tipo IN ['Glaciar', 'Nevado']
OPTIONAL MATCH (g)-[:MONITOREADO_CON]->(m:MetodoMonitoreo)
WHERE m.nombre IN $balance
OPTIONAL MATCH (g)-[:PRESENTA_FENOMENO]->(f:Fenomeno)
WHERE f.nombre = 'Retroceso glaciar'
OPTIONAL MATCH (d:Documento)-[:MENCIONA]->(g), (d)-[:USA_METODO]->(mb:MetodoMonitoreo)
WHERE mb.nombre IN $balance
WITH l, g, ra, count(DISTINCT m) AS n_medido, count(DISTINCT f) AS n_reportado, count(DISTINCT d) AS n_doc
WHERE n_medido + n_reportado + n_doc > 0
RETURN g.nombre AS glaciar,
       CASE WHEN n_medido > 0 THEN 'medido (balance de masa)'
            WHEN n_reportado > 0 THEN 'retroceso reportado'
            ELSE 'en documento con balance de masa' END AS retroceso,
       l.nombre AS laguna, l.nivel_max AS nivel, ra.documento AS documento, ra.pagina AS pagina, ra.evidencia AS evidencia
ORDER BY retroceso DESC, glaciar"""),
    "P6": dict(
        titulo="¿Qué métodos de monitoreo se aplicaron a lagunas en riesgo y qué institución realizó cada estudio?",
        params=P_RIESGO,
        cypher="""
MATCH (l:Laguna)-[r:MONITOREADO_CON]->(m:MetodoMonitoreo)
WHERE l.nivel_max IN $niveles
OPTIONAL MATCH (i:Institucion)-[:INTERVIENE_EN]->(l)
OPTIONAL MATCH (doc:Documento)-[:PUBLICADO_POR]->(pub:Institucion)
WHERE doc.id = r.documento
RETURN l.nombre AS laguna, m.nombre AS metodo, collect(DISTINCT i.nombre) AS instituciones_en_texto,
       collect(DISTINCT pub.nombre) AS institucion_del_documento,
       r.documento AS documento, r.pagina AS pagina, r.evidencia AS evidencia
ORDER BY laguna, metodo"""),
    "P7": dict(
        titulo="¿En qué subcuencas coinciden lagunas en riesgo y contaminación por drenaje ácido de roca?",
        params={**P_RIESGO, "fenomeno": "Drenaje ácido de roca"},
        cypher="""
MATCH (l:Laguna)-[:UBICADA_EN]->(c)
WHERE l.nivel_max IN $niveles AND c.hidro = true AND c.grande = false
MATCH (f:Fenomeno)
WHERE f.nombre = $fenomeno
OPTIONAL MATCH (c)-[r1:PRESENTA_FENOMENO]->(f)
OPTIONAL MATCH (x)-[:UBICADA_EN]->(c), (x)-[r2:PRESENTA_FENOMENO]->(f)
WITH c, l, r1, x, r2
WHERE r1 IS NOT NULL OR r2 IS NOT NULL
RETURN c.nombre AS subcuenca, collect(DISTINCT l.nombre) AS lagunas_en_riesgo,
       collect(DISTINCT x.nombre) AS con_drenaje_acido,
       collect(DISTINCT coalesce(r1.documento, r2.documento)) AS documentos,
       collect(DISTINCT coalesce(r1.evidencia, r2.evidencia)) AS evidencias
ORDER BY subcuenca"""),
    "P8": dict(
        titulo="¿Qué bofedales están en subcuencas con glaciares en retroceso y qué servicios ecosistémicos brindan?",
        params={},
        cypher="""
MATCH (g)-[:PRESENTA_FENOMENO]->(f:Fenomeno)
WHERE f.nombre = 'Retroceso glaciar' AND g.tipo IN ['Glaciar', 'Nevado', 'Cuenca', 'Quebrada']
MATCH (b:Bofedal)-[rb:UBICADA_EN]->(c)
WHERE c.hidro = true AND c.grande = false AND b.dudoso = false AND (c.id = g.id OR EXISTS { MATCH (g)-[:UBICADA_EN]->(c) })
OPTIONAL MATCH (b)-[:PROVEE_SERVICIO]->(s:ServicioEcosistemico)
OPTIONAL MATCH (e:Ecosistema)-[:PROVEE_SERVICIO]->(s2:ServicioEcosistemico)
WHERE e.k = 'bofedal'
RETURN b.nombre AS bofedal, c.nombre AS subcuenca, collect(DISTINCT g.nombre) AS en_retroceso,
       collect(DISTINCT s.nombre) AS servicios_del_bofedal, collect(DISTINCT s2.nombre) AS servicios_de_los_bofedales,
       rb.documento AS documento, rb.pagina AS pagina, rb.evidencia AS evidencia
ORDER BY subcuenca, bofedal"""),
    "P9": dict(
        titulo="¿Cómo cambió la superficie glaciar de una cordillera entre los inventarios de 2018 y 2023?",
        params={"cordillera": "blanca"},
        cypher="""
MATCH (i:Inventario)-[:INVENTARIO_DE]->(c:Cordillera {k: $cordillera})
MATCH (i)-[:TIENE_MEDICION]->(m:Medicion)
WHERE m.propiedad = 'registraSuperficieGlaciar' AND m.valor IS NOT NULL
RETURN i.anio AS anio, i.anio_imagenes AS anio_imagenes, m.valor AS km2, m.confianza AS confianza,
       m.documento AS documento, m.pagina AS pagina, m.evidencia AS evidencia
ORDER BY anio"""),
    "P10": dict(
        titulo="¿Qué lagunas en riesgo cuentan con obras de seguridad, en qué estado están y qué poblados protegen?",
        params=P_RIESGO,
        cypher="""
MATCH (l:Laguna)-[r:TIENE_OBRA]->(o:ObraSeguridad)
WHERE l.nivel_max IN $niveles
OPTIONAL MATCH (l)-[re:ESTADO_OBRA]->(e:EstadoObra)
WHERE re.obra = o.nombre
OPTIONAL MATCH (l)-[:AMENAZA_A]->(p:CentroPoblado)
WHERE p.dudoso = false
RETURN l.nombre AS laguna, l.nivel_max AS nivel, o.nombre AS obra, collect(DISTINCT e.nombre) AS estado,
       collect(DISTINCT p.nombre) AS poblados_protegidos, r.documento AS documento, r.pagina AS pagina, r.evidencia AS evidencia
ORDER BY laguna, obra"""),
}
EXTRAS = {
    "E1_negacion_sin_obras": dict(
        titulo="Negación · ¿Qué lagunas en riesgo (alto o muy alto) NO tienen obras de seguridad registradas?",
        params=P_RIESGO, cypher="""
MATCH (l:Laguna)-[r:TIENE_NIVEL_RIESGO]->(n:NivelRiesgo)
WHERE n.nombre IN $niveles AND NOT EXISTS { MATCH (l)-[:TIENE_OBRA]->(:ObraSeguridad) }
RETURN l.nombre AS laguna, n.nombre AS nivel, l.departamentos AS departamentos, r.documento AS documento, r.pagina AS pagina, r.evidencia AS evidencia
ORDER BY nivel DESC, laguna"""),
    "E2_negacion_sin_monitoreo": dict(
        titulo="Negación · ¿Qué lagunas en riesgo no tienen ningún método de monitoreo registrado?",
        params=P_RIESGO, cypher="""
MATCH (l:Laguna)
WHERE l.nivel_max IN $niveles AND NOT EXISTS { MATCH (l)-[:MONITOREADO_CON]->(:MetodoMonitoreo) }
RETURN l.nombre AS laguna, l.nivel_max AS nivel, l.departamentos AS departamentos
ORDER BY laguna"""),
    "E3_conteo_departamento": dict(
        titulo="Conteo · ¿Cuántas lagunas en riesgo hay por departamento?",
        params=P_RIESGO, cypher="""
MATCH (l:Laguna)
WHERE l.nivel_max IN $niveles
UNWIND l.departamentos AS departamento
RETURN departamento, count(DISTINCT l) AS lagunas_en_riesgo, collect(DISTINCT l.nombre) AS lagunas
ORDER BY lagunas_en_riesgo DESC"""),
    "E4_conteo_cordillera": dict(
        titulo="Conteo con varios saltos · ¿Qué cordillera tiene más lagunas en riesgo?",
        params=P_RIESGO, cypher="""
MATCH (l:Laguna)-[:UBICADA_EN*1..3]->(c:Cordillera)
WHERE l.nivel_max IN $niveles
RETURN c.nombre AS cordillera, count(DISTINCT l) AS lagunas_en_riesgo, collect(DISTINCT l.nombre) AS lagunas
ORDER BY lagunas_en_riesgo DESC"""),
    "E5_ranking_volumen": dict(
        titulo="Ranking · Las 5 lagunas de Áncash con mayor volumen",
        params={"departamento": "Áncash"}, cypher="""
MATCH (l:Laguna)-[:TIENE_MEDICION]->(m:Medicion)
WHERE m.propiedad = 'tieneVolumen' AND m.valor IS NOT NULL AND $departamento IN l.departamentos
RETURN l.nombre AS laguna, max(m.valor) AS volumen_m3, collect(DISTINCT m.documento) AS documentos
ORDER BY volumen_m3 DESC LIMIT 5"""),
    "E6_linea_tiempo": dict(
        titulo="Línea de tiempo · Eventos registrados antes de 1980",
        params={"hasta": 1980}, cypher="""
MATCH (e:Evento)
WHERE e.anio < $hasta
OPTIONAL MATCH (e)-[r:OCURRIO_EN]->(x)
OPTIONAL MATCH (e)-[:AFECTO_A]->(p)
RETURN e.anio AS anio, e.nombre AS evento, collect(DISTINCT x.nombre) AS lugar, collect(DISTINCT p.nombre) AS afecto_a,
       collect(DISTINCT r.documento) AS documentos
ORDER BY anio"""),
    "E7_instituciones": dict(
        titulo="Conteo · Instituciones que intervienen en más lagunas",
        params={}, cypher="""
MATCH (i:Institucion)-[:INTERVIENE_EN]->(l:Laguna)
RETURN i.nombre AS institucion, count(DISTINCT l) AS lagunas, collect(DISTINCT l.nombre) AS ejemplos
ORDER BY lagunas DESC LIMIT 10"""),
    "E8_ficha": dict(
        titulo="Ficha · Todo lo que se sabe de la laguna Palcacocha (con fuente)",
        params={"k": "palcacocha"}, cypher="""
MATCH (a:Entidad {k: $k})-[r]-(b)
WHERE r.pred <> 'mencionaA'
RETURN a.tipo AS tipo, r.pred AS relacion, b.nombre AS con, b.tipo AS tipo_con, r.n_docs AS n_docs,
       r.documento AS documento, r.pagina AS pagina, r.evidencia AS evidencia
ORDER BY n_docs DESC LIMIT 40"""),
    "E9_comencion": dict(
        titulo="Co-mención · Documentos que hablan a la vez de Palcacocha y Huaraz",
        params={"a": "palcacocha", "b": "huaraz"}, cypher="""
MATCH (d:Documento)-[:MENCIONA]->(a:Entidad {k: $a}), (d)-[:MENCIONA]->(b:Entidad {k: $b})
RETURN DISTINCT d.id AS documento, d.titulo AS titulo, d.fuente AS fuente, d.anio AS anio
ORDER BY anio DESC"""),
    "E10_camino": dict(
        titulo="Camino · ¿Cómo se conecta el nevado Huascarán con Yungay? (sin pasar por documentos)",
        params={"a": "huascaran", "b": "yungay"}, solo_neo4j=True, cypher="""
MATCH (a:Entidad {k: $a}), (b:Entidad {k: $b})
MATCH p = shortestPath((a)-[*..5]-(b))
WHERE ALL(r IN relationships(p) WHERE NOT type(r) IN ['MENCIONA', 'USA_METODO', 'PUBLICADO_POR'])
RETURN [n IN nodes(p) | n.nombre] AS camino, [r IN relationships(p) | r.pred] AS relaciones,
       [r IN relationships(p) | r.documento] AS documentos
LIMIT 3"""),
}


def comparar_superficie(df, a1=2018, a2=2023):
    """P9: elige el valor de cada año (verificado primero; sin atípicos) y calcula el cambio."""
    if df is None or not len(df):
        return pd.DataFrame()
    d = df.dropna(subset=["anio", "km2"]).copy()
    d["anio"] = d.anio.astype(int)
    med = d.km2.median()
    d = d[(d.km2 >= 0.4 * med) & (d.km2 <= 2.5 * med)].sort_values(["anio", "confianza"], ascending=[True, False])
    serie = d.drop_duplicates("anio")
    if not len(serie):
        return pd.DataFrame()
    cerca = lambda a: serie.iloc[(serie.anio - a).abs().argsort().iloc[0]]
    x, y = cerca(a1), cerca(a2)
    if x.anio == y.anio and len(serie) > 1:
        antes = serie[serie.anio < y.anio]
        x = antes.iloc[-1] if len(antes) else serie.iloc[0]
    nota = "" if (x.anio, y.anio) == (a1, a2) else f"años pedidos {a1}/{a2}; se usan los más cercanos con dato"
    if y.km2 > x.km2:
        nota += " ⚠️ aumento: revisar fuentes"
    return pd.DataFrame([{"anio_1": int(x.anio), "km2_1": x.km2, "anio_2": int(y.anio), "km2_2": y.km2,
                          "cambio_km2": round(y.km2 - x.km2, 2), "cambio_%": round(100 * (y.km2 - x.km2) / x.km2, 1),
                          "fuente_1": f"{x.documento} p.{x.pagina}", "fuente_2": f"{y.documento} p.{y.pagina}", "nota": nota.strip()}])


COMPLETO = {   # qué columna debe tener datos para considerar que todos los saltos se cumplieron
    "P3": "estudios", "P4": "glaciar_origen", "P6": "instituciones_en_texto", "P10": "poblados_protegidos",
}


def evaluar(ejecutar, consultas=None):
    """Corre las consultas con la función `ejecutar(cypher, **params)` y califica ✅/🟡/❌."""
    filas, resp = [], {}
    for k, c in (consultas or CONSULTAS).items():
        t0 = time.time()
        try:
            d = ejecutar(c["cypher"], **c.get("params", {}))
            err = ""
        except Exception as e:
            d, err = pd.DataFrame(), str(e)[:200]
        resp[k] = d
        col = COMPLETO.get(k)
        lleno = (not col or (col in d and d[col].map(lambda v: len(v) if isinstance(v, (list, tuple)) else bool(v)).astype(bool).any()))
        estado = "❌" if err or not len(d) else ("✅" if lleno else "🟡")
        filas.append({"pregunta": k, "enunciado": c["titulo"], "estado": estado, "filas": len(d),
                      "segundos": round(time.time() - t0, 2), "error": err})
    return pd.DataFrame(filas), resp


def archivo_cypher(ruta):
    """Guarda todas las consultas en un .cypher (para el repositorio y para pegarlas en Neo4j Browser)."""
    with open(ruta, "w", encoding="utf-8") as f:
        f.write("// Consultas Cypher · Enterprise Brain Glaciares (G8)\n// Parámetros de ejemplo en cada bloque (:param en Neo4j Browser)\n\n")
        for k, c in {**CONSULTAS, **EXTRAS}.items():
            f.write(f"// {k} · {c['titulo']}\n")
            for p, v in c.get("params", {}).items():
                f.write(f":param {p} => {json.dumps(v, ensure_ascii=False)};\n")
            f.write(c["cypher"].strip() + ";\n\n")


# ======================================================================
# 4. Capa 2 — Grafo RDF (ontología + datos) y SPARQL: deducción y validación
# ======================================================================
def _local(nid):
    t, _, resto = str(nid).partition(":")
    resto = resto or t
    return f"{t}/{re.sub(r'[^A-Za-z0-9_./-]+', '_', resto)}" if resto != t else "doc/" + re.sub(r"[^A-Za-z0-9_.-]+", "_", t)


def construir_rdf(N, R, ontologia=None, log=print):
    """Grafo RDF: cada nodo es un individuo de su clase; cada relación es un triple y, además, un eb:Hecho
    (reificación) con su procedencia (prov:wasDerivedFrom documento, página, evidencia, confianza)."""
    from rdflib import Graph, Literal, URIRef, BNode
    from rdflib.namespace import RDF, RDFS, XSD
    ns = _ns()
    eb, ent, prov, sosa = ns["eb"], ns["ent"], ns["prov"], ns["sosa"]
    g = Graph()
    for k, v in ns.items():
        g.bind(k, v)
    if ontologia is not None:
        g += ontologia
    t0 = time.time()
    iri = {}
    tipo_de = dict(zip(N.id, N.tipo))
    for r in N.to_dict("records"):
        t = r["tipo"]
        u = ent[_local(r["id"])] if t != "Documento" else ent["doc/" + re.sub(r"[^A-Za-z0-9_.-]+", "_", r["id"])]
        if t == "NivelRiesgo":
            u = eb["nivel_" + clave("x", r["nombre"]).split(":")[1]]
        iri[r["id"]] = u
        cls = {"Cuenca": "Cuenca", "Quebrada": "Quebrada"}.get(t, t)
        g.add((u, RDF.type, eb[cls] if cls in CLASES else eb["RasgoGeografico"] if cls == "Rio" else eb[cls]))
        g.add((u, RDFS.label, Literal(r["nombre"], lang="es")))
        if r.get("anio") == r.get("anio") and r.get("anio") is not None:
            g.add((u, eb.anio, Literal(int(r["anio"]), datatype=XSD.integer)))
        if t == "Inventario" and r.get("anio_imagenes") == r.get("anio_imagenes") and r.get("anio_imagenes") is not None:
            g.add((u, eb.anioImagenes, Literal(int(r["anio_imagenes"]), datatype=XSD.integer)))
        if t == "Documento":
            g.add((u, ns["dcterms"].identifier, Literal(r["id"])))
            if r.get("fuente"):
                g.add((u, eb.fuente, Literal(r["fuente"])))
        if t == "Medicion":
            g.add((u, eb.propiedadMedida, Literal(r.get("propiedad", ""))))
            g.add((u, sosa.hasSimpleResult, Literal(r.get("texto", ""))))
            if r.get("valor") == r.get("valor") and r.get("valor") is not None:
                g.add((u, eb.valor, Literal(float(r["valor"]), datatype=XSD.double)))
            if r.get("unidad"):
                g.add((u, eb.unidad, Literal(r["unidad"])))
    n_hechos = 0
    for r in R.to_dict("records"):
        s, o = iri.get(r["s"]), iri.get(r["o"])
        if s is None or o is None:
            continue
        p = eb[PRED_REL_INV.get(r["tipo"], r["pred"])]
        g.add((s, p, o))
        if r["tipo"] == "TIENE_MEDICION":
            g.add((o, sosa.hasFeatureOfInterest, s))
        doc = r.get("documento")
        if r["tipo"] in ("MENCIONA", "USA_METODO", "PUBLICADO_POR") or not isinstance(doc, str) or not doc:
            continue
        h = ent["hecho/" + r["fid"]]
        g.add((h, RDF.type, eb.Hecho))
        g.add((h, RDF.subject, s)); g.add((h, RDF.predicate, p)); g.add((h, RDF.object, o))
        g.add((h, prov.wasDerivedFrom, ent["doc/" + re.sub(r"[^A-Za-z0-9_.-]+", "_", doc)]))
        if r.get("pagina") == r.get("pagina") and r.get("pagina") is not None:
            g.add((h, eb.pagina, Literal(int(r["pagina"]), datatype=XSD.integer)))
        if isinstance(r.get("evidencia"), str):
            g.add((h, eb.evidencia, Literal(r["evidencia"])))
        if r.get("confianza") == r.get("confianza") and r.get("confianza") is not None:
            g.add((h, eb.confianza, Literal(float(r["confianza"]), datatype=XSD.double)))
        if isinstance(r.get("obra"), str) and r["obra"]:
            g.add((h, RDFS.comment, Literal("obra: " + r["obra"])))
        n_hechos += 1
    log(f"RDF: {len(g):,} triples · {n_hechos:,} hechos con procedencia · {time.time() - t0:.0f} s")
    return g


PREFIJOS = """PREFIX eb: <https://w3id.org/g8-glaciares/ontologia#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX sosa: <http://www.w3.org/ns/sosa/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dcterms: <http://purl.org/dc/terms/>
"""
SPARQL = {
    "S1_P2_deduccion": dict(
        titulo="P2 con deducción · Lagunas con riesgo muy alto en Áncash (el departamento puede deducirse subiendo por ubicadaEn+)",
        sparql="""
SELECT DISTINCT ?laguna ?como ?documento ?pagina ?evidencia WHERE {
  ?l a eb:Laguna ; rdfs:label ?laguna ; eb:tieneNivelRiesgo eb:nivel_muy_alto .
  ?d a eb:Departamento ; rdfs:label ?dn . FILTER(STR(?dn) = "Áncash")
  ?l eb:ubicadaEn+ ?d .
  # si el texto ubica la laguna en OTRO departamento y nunca en este, se descarta
  FILTER ( EXISTS { ?l eb:ubicadaEn ?d } || NOT EXISTS { ?l eb:ubicadaEn ?d2 . ?d2 a eb:Departamento } )
  BIND ( IF(EXISTS { ?l eb:ubicadaEn ?d }, "dicho en el texto", "deducido por la jerarquía") AS ?como )
  ?h a eb:Hecho ; rdf:subject ?l ; rdf:predicate eb:tieneNivelRiesgo ; rdf:object eb:nivel_muy_alto ;
     prov:wasDerivedFrom ?doc .
  OPTIONAL { ?h eb:pagina ?pagina } OPTIONAL { ?h eb:evidencia ?evidencia }
  OPTIONAL { ?doc dcterms:identifier ?documento }
} ORDER BY ?laguna"""),
    "S2_deduccion_departamentos": dict(
        titulo="Deducción · ¿Cuántas lagunas tienen departamento dicho en el texto y cuántas solo deducido (ubicadaEn es transitiva)?",
        sparql="""
SELECT ?como (COUNT(DISTINCT ?l) AS ?lagunas) WHERE {
  ?l a eb:Laguna .
  { ?l eb:ubicadaEn ?d . ?d a eb:Departamento . BIND("dicho en el texto" AS ?como) }
  UNION
  { ?l eb:ubicadaEn/eb:ubicadaEn+ ?d . ?d a eb:Departamento .
    FILTER NOT EXISTS { ?l eb:ubicadaEn ?d2 . ?d2 a eb:Departamento } BIND("solo deducido" AS ?como) }
} GROUP BY ?como"""),
    "S3_jerarquia": dict(
        titulo="Jerarquía de clases · Elementos geográficos por clase (rdf:type/rdfs:subClassOf* geo:Feature)",
        sparql="""
SELECT ?clase (COUNT(?x) AS ?n) WHERE {
  ?x a ?c . ?c rdfs:subClassOf* geo:Feature . ?c rdfs:label ?clase .
} GROUP BY ?clase ORDER BY DESC(?n)"""),
    "S4_procedencia": dict(
        titulo="Procedencia (PROV-O) · ¿De dónde sale que Palcacocha amenaza a Huaraz?",
        sparql="""
SELECT ?documento ?titulo ?pagina ?evidencia ?confianza WHERE {
  ?l rdfs:label ?ln . FILTER(STR(?ln) = "Palcacocha")
  ?p rdfs:label ?pn . FILTER(STR(?pn) = "Huaraz")
  ?h rdf:subject ?l ; rdf:predicate eb:amenazaA ; rdf:object ?p ; prov:wasDerivedFrom ?doc .
  OPTIONAL { ?doc rdfs:label ?titulo } OPTIONAL { ?h eb:pagina ?pagina } OPTIONAL { ?h eb:evidencia ?evidencia }
  OPTIONAL { ?h eb:confianza ?confianza } OPTIONAL { ?doc dcterms:identifier ?documento }
} ORDER BY DESC(?confianza)"""),
    "S5_negacion": dict(
        titulo="Negación · Lagunas con riesgo muy alto sin obra de seguridad",
        sparql="""
SELECT ?laguna WHERE {
  ?l a eb:Laguna ; rdfs:label ?laguna ; eb:tieneNivelRiesgo eb:nivel_muy_alto .
  FILTER NOT EXISTS { ?l eb:tieneObraSeguridad ?o }
} ORDER BY ?laguna"""),
    "S6_P4_eventos": dict(
        titulo="P4 · Eventos en lagunas, glaciar de origen y poblados afectados",
        sparql="""
SELECT ?evento ?anio ?laguna (GROUP_CONCAT(DISTINCT ?g2; separator=", ") AS ?glaciar_origen)
       (GROUP_CONCAT(DISTINCT ?p2; separator=", ") AS ?afecto_a) WHERE {
  ?e a eb:Evento ; rdfs:label ?evento ; eb:ocurrioEn ?l . ?l a eb:Laguna ; rdfs:label ?laguna .
  OPTIONAL { ?e eb:anio ?anio }
  OPTIONAL { ?l eb:alimentadaPor ?g . ?g rdfs:label ?gn }
  OPTIONAL { ?e eb:afectoA ?p . ?p rdfs:label ?pn }
  BIND ( COALESCE(STR(?gn), "") AS ?g2 )
  BIND ( COALESCE(STR(?pn), "") AS ?p2 )
} GROUP BY ?evento ?anio ?laguna ORDER BY ?anio"""),
    "S7_P9_superficie": dict(
        titulo="P9 · Superficie glaciar de la Cordillera Blanca por inventario (SOSA: Medicion = observación)",
        sparql="""
SELECT ?inventario ?anio ?km2 ?documento ?pagina WHERE {
  ?c a eb:Cordillera ; rdfs:label ?cn . FILTER(STR(?cn) = "Blanca")
  ?i eb:inventarioDe ?c ; rdfs:label ?inventario ; eb:anio ?anio ; eb:tieneMedicion ?m .
  ?m eb:propiedadMedida "registraSuperficieGlaciar" ; eb:valor ?km2 .
  ?h rdf:object ?m ; prov:wasDerivedFrom ?doc .
  OPTIONAL { ?doc dcterms:identifier ?documento } OPTIONAL { ?h eb:pagina ?pagina }
} ORDER BY ?anio"""),
}
VALIDACION = {
    "amenazaA hacia algo que no es unidad territorial": """
SELECT (COUNT(*) AS ?n) WHERE { ?s eb:amenazaA ?o . FILTER NOT EXISTS { ?o a/rdfs:subClassOf* eb:UnidadTerritorial } }""",
    "ubicadaEn hacia algo que no es un lugar (geo:Feature)": """
SELECT (COUNT(*) AS ?n) WHERE { ?s eb:ubicadaEn ?o . FILTER NOT EXISTS { ?o a/rdfs:subClassOf* geo:Feature } }""",
    "nivel de riesgo en algo que no es laguna": """
SELECT (COUNT(*) AS ?n) WHERE { ?s eb:tieneNivelRiesgo ?o . FILTER NOT EXISTS { ?s a eb:Laguna } }""",
    "individuos en clases disjuntas (rasgo geográfico y unidad territorial a la vez)": """
SELECT (COUNT(DISTINCT ?x) AS ?n) WHERE { ?x a/rdfs:subClassOf* eb:RasgoGeografico . ?x a/rdfs:subClassOf* eb:UnidadTerritorial }""",
    "hechos sin documento de origen": """
SELECT (COUNT(*) AS ?n) WHERE { ?h a eb:Hecho . FILTER NOT EXISTS { ?h prov:wasDerivedFrom ?d } }""",
}


def sparql(g, consulta):
    res = g.query(PREFIJOS + consulta)
    filas = [[(v.toPython() if v is not None else None) for v in fila] for fila in res]
    return pd.DataFrame(filas, columns=[str(v) for v in res.vars])


def validar_rdf(g):
    """Reglas de la ontología revisadas con SPARQL: 0 = sin violaciones."""
    return pd.DataFrame([{"regla": k, "violaciones": int(sparql(g, q).iloc[0, 0])} for k, q in VALIDACION.items()])


def archivo_sparql(ruta):
    with open(ruta, "w", encoding="utf-8") as f:
        f.write("# Consultas SPARQL · Enterprise Brain Glaciares (G8)\n" + PREFIJOS + "\n")
        for k, c in SPARQL.items():
            f.write(f"# {k} · {c['titulo']}\n{c['sparql'].strip()}\n\n")
        f.write("# ---- Validación (deben dar 0) ----\n")
        for k, q in VALIDACION.items():
            f.write(f"# {k}\n{q.strip()}\n\n")


# ======================================================================
# 5. Capa 3 — Vistas interactivas (Pyvis), camino sin conexión y archivos de respaldo
# ======================================================================
COLORES = {"Laguna": "#2b7bba", "Glaciar": "#8fd3f4", "Nevado": "#b8e0f0", "CentroPoblado": "#e4572e",
           "Cordillera": "#6c757d", "Cuenca": "#3fa34d", "Quebrada": "#5cb85c", "Departamento": "#adb5bd",
           "Provincia": "#ced4da", "Distrito": "#dee2e6", "Evento": "#d62828", "ObraSeguridad": "#f4a261",
           "NivelRiesgo": "#9d0208", "EstadoObra": "#e9c46a", "Institucion": "#6a4c93", "MetodoMonitoreo": "#8338ec",
           "Medicion": "#90be6d", "Inventario": "#43aa8b", "Documento": "#495057", "Bofedal": "#2a9d8f",
           "Fenomeno": "#bc6c25", "ServicioEcosistemico": "#52b788", "Ecosistema": "#74c69d", "Especie": "#95d5b2"}
SIN_DOCS = ("MENCIONA", "USA_METODO", "PUBLICADO_POR")


def subgrafo(N, R, centro_k, saltos=1, max_nodos=60, sin_docs=True, tipos_rel=None):
    """Vecindario de una entidad (por nombre normalizado) a `saltos` de distancia."""
    Rx = R[~R.tipo.isin(SIN_DOCS)] if sin_docs else R
    if tipos_rel:
        Rx = Rx[Rx.tipo.isin(tipos_rel)]
    ids = set(N[N.k == norm(centro_k)].id)
    if not ids:
        return N.iloc[0:0], R.iloc[0:0]
    frontera = set(ids)
    for _ in range(saltos):
        e = Rx[Rx.s.isin(frontera) | Rx.o.isin(frontera)]
        e = e.sort_values("n_docs", ascending=False, na_position="last")
        nuevos = (set(e.s) | set(e.o)) - ids
        frontera = set(list(nuevos)[: max(0, max_nodos - len(ids))])
        ids |= frontera
    E = Rx[Rx.s.isin(ids) & Rx.o.isin(ids)]
    return N[N.id.isin(ids)], E


def vista_interactiva(Nn, Rr, ruta, titulo="", centro_k=None, alto="680px"):
    """Página HTML interactiva (zoom, arrastrar, pasar el mouse = evidencia y fuente). No necesita internet."""
    from pyvis.network import Network
    net = Network(height=alto, width="100%", directed=True, notebook=False, cdn_resources="in_line", bgcolor="#ffffff")
    net.barnes_hut(gravity=-6000, spring_length=140)
    for r in Nn.to_dict("records"):
        es_centro = centro_k and r["k"] == norm(centro_k)
        etiqueta = r["nombre"] if r["tipo"] != "Medicion" else str(r.get("texto", ""))[:28]
        info = f"{r['tipo']}: {r['nombre']}"
        if isinstance(r.get("departamentos"), list) and r["departamentos"]:
            info += f"\nDepartamento: {', '.join(r['departamentos'])}"
        if isinstance(r.get("nivel_max"), str):
            info += f"\nRiesgo máximo citado: {r['nivel_max']}"
        net.add_node(r["id"], label=etiqueta, title=info, color=COLORES.get(r["tipo"], "#999999"),
                     size=34 if es_centro else 20, shape="dot", font={"size": 26 if es_centro else 20, "strokeWidth": 3})
    for r in Rr.to_dict("records"):
        ev = str(r.get("evidencia") or "")[:220]
        doc = r.get("documento") or ""
        pag = r.get("pagina")
        tip = f"{r['pred']}\nFuente: {doc}" + (f", p. {int(pag)}" if pag == pag and pag is not None else "") + (f"\n«{ev}»" if ev else "")
        net.add_edge(r["s"], r["o"], label=r["pred"], title=tip, arrows="to", font={"size": 13, "align": "middle", "strokeWidth": 3})
    html = net.generate_html()
    tipos = sorted(set(Nn.tipo))
    leyenda = " ".join(f"<span style='display:inline-block;margin:2px 10px 2px 0'><span style='display:inline-block;width:12px;"
                       f"height:12px;border-radius:6px;background:{COLORES.get(t, '#999')};vertical-align:middle'></span> {t}</span>"
                       for t in tipos)
    cabecera = (f"<div style='font-family:sans-serif;margin:8px'><h3 style='margin:4px 0'>{titulo}</h3>"
                f"<div style='font-size:13px'>{leyenda}</div><div style='font-size:12px;color:#555'>"
                f"Rueda del mouse = zoom · arrastrar = mover · pasar el mouse por una flecha = fuente y evidencia</div></div>")
    html = html.replace("<body>", "<body>" + cabecera, 1)
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(html)
    return ruta


def vista_riesgo(N, R, departamento="Áncash", niveles=("Muy alto",)):
    """Subgrafo de riesgo: lagunas con ese nivel en el departamento + poblados amenazados, obras, glaciares y estado."""
    lag = N[(N.tipo == "Laguna") & N.nivel_max.isin(list(niveles)) &
            N.departamentos.map(lambda d: isinstance(d, list) and departamento in d).astype(bool)]
    ids = set(lag.id)
    E = R[R.s.isin(ids) & R.tipo.isin(["AMENAZA_A", "TIENE_OBRA", "ALIMENTADA_POR", "TIENE_NIVEL_RIESGO", "ESTADO_OBRA"])]
    E = E[~E.o.isin(set(N[N.dudoso == True].id))]
    ids |= set(E.o)
    return N[N.id.isin(ids)], E


def camino_local(N, R, a, b, max_saltos=5):
    """Camino más corto entre dos entidades sin pasar por documentos (respaldo sin Neo4j)."""
    import networkx as nx
    Rx = R[~R.tipo.isin(SIN_DOCS)]
    G_ = nx.Graph()
    for r in Rx.itertuples():
        G_.add_edge(r.s, r.o, pred=r.pred, doc=getattr(r, "documento", ""), pag=getattr(r, "pagina", None))
    nombre = dict(zip(N.id, N.nombre))
    A = [i for i in N[N.k == norm(a)].id if i in G_]
    B = [i for i in N[N.k == norm(b)].id if i in G_]
    mejor = None
    for x in A:
        for y in B:
            try:
                p = nx.shortest_path(G_, x, y)
            except nx.NetworkXNoPath:
                continue
            if len(p) - 1 <= max_saltos and (mejor is None or len(p) < len(mejor)):
                mejor = p
    if not mejor:
        return pd.DataFrame()
    return pd.DataFrame([{"desde": nombre[u], "relacion": G_[u][v]["pred"], "hacia": nombre[v],
                          "documento": G_[u][v]["doc"], "pagina": G_[u][v]["pag"]} for u, v in zip(mejor, mejor[1:])])


def exportar_csv_neo4j(N, R, carpeta):
    """nodos.csv y relaciones.csv: sirven para el 'Importador' de Neo4j Aura (sin código) y como respaldo."""
    os.makedirs(carpeta, exist_ok=True)
    n = N.copy()
    n["departamentos"] = n.departamentos.map(lambda d: ";".join(d) if isinstance(d, list) else "")
    n.to_csv(os.path.join(carpeta, "nodos.csv"), index=False, encoding="utf-8")
    R.to_csv(os.path.join(carpeta, "relaciones.csv"), index=False, encoding="utf-8")
    return os.path.join(carpeta, "nodos.csv"), os.path.join(carpeta, "relaciones.csv")


README_GRAFO = """# Ontología y grafo de conocimiento · Glaciares (semana 7)

| Archivo | Qué es |
|---|---|
| `glaciares.owl` | Ontología OWL (abrir en Protégé): clases, jerarquía, propiedades con dominio y rango, alineada a GeoSPARQL, SOSA/SSN y PROV-O |
| `grafo_glaciares.ttl.gz` | Grafo RDF completo (ontología + datos + procedencia de cada hecho) en Turtle comprimido |
| `consultas.cypher` | Las 10 preguntas de competencia y 10 consultas extra en Cypher (Neo4j) |
| `consultas.sparql` | Consultas SPARQL: deducción, jerarquía, procedencia, negación y reglas de validación |
| `nodos.csv`, `relaciones.csv` | El grafo en CSV (importador de Neo4j Aura y respaldo) |
| `vista_*.html` | Vistas interactivas del grafo (abrir en el navegador; funcionan sin internet) |
| `respuestas_cypher/` | Resultado de cada consulta, con documento, página y evidencia |
| `reporte_grafo.md` | Conteos, evaluación de las preguntas y validación |

## Modelo (grafo de propiedades en Neo4j)
- Cada nodo tiene la etiqueta `:Entidad` y la de su tipo (`:Laguna`, `:Glaciar`, `:CentroPoblado`, `:Documento`, `:Medicion`…),
  con `id`, `nombre` y `k` (nombre sin tildes ni mayúsculas, para buscar).
- Cada relación guarda su **procedencia**: `documento`, `pagina`, `evidencia`, `confianza`, `n_docs`, `metodos`.
- Los valores numéricos (volumen, área, altitud, superficie, conteos de inventario) son nodos `:Medicion` (sosa:Observation)
  con valor, unidad, año y fuente.
- Propiedades derivadas (explicadas): `departamentos` (el que dice el texto manda; si no hay, se deduce por la jerarquía),
  `nivel_max` y `en_riesgo` (nivel de riesgo más alto citado), `hidro`/`grande`/`n_poblados` (subcuencas), `dudoso`
  (nombres que GLiNER marcó como poblado o bofedal pero son nevados, lagunas o departamentos).
"""


print("Grafo cargado · versión", VERSION)

Grafo cargado · versión 1.0-grafo


## 4. Cargar los hechos de la Persona 3 y armar el grafo

In [ ]:
cons = pd.read_csv(os.path.join(CARPETA_TRIP, "tripletas_consolidadas.csv"), dtype={"paper_id": str, "subject_id": str, "object_id": str}, low_memory=False)
entidades = pd.read_csv(os.path.join(CARPETA_TRIP, "entidades.csv"))
papers = pd.read_csv(os.path.join(CARPETA_CORPUS, "papers.csv"), dtype=str)
print(f"Hechos: {len(cons):,} · entidades: {len(entidades):,} · documentos: {len(papers)}")
print("Hechos verificados por el equipo (curaduría):", int(cons.metodos.astype(str).str.contains("curado").sum()))

# curaduría negativa: hechos que el equipo revisó y marcó como errores (curaduria/hechos_rechazados.csv)
RUTA_RECH = os.path.join(BASE, "curaduria", "hechos_rechazados.csv")
if os.path.exists(RUTA_RECH):
    rechazos = pd.read_csv(RUTA_RECH, dtype=str)
    cons, quitados = aplicar_rechazos(cons, rechazos)
    with pd.option_context("display.max_colwidth", 70):
        display(quitados[["subject", "predicate", "object", "paper_id", "page_start"]].head(30))
else:
    print("Sin lista de hechos rechazados:", RUTA_RECH)

GRAFO = preparar_grafo(cons, entidades, papers, CONFIG["CONF_MIN"])
N, R = GRAFO["nodos"], GRAFO["rels"]
display(pd.Series(GRAFO["resumen"]["por_tipo"], name="nodos").to_frame().T)
display(pd.Series(GRAFO["resumen"]["por_relacion"], name="relaciones").to_frame().T)

Hechos: 37,729 · entidades: 11,381 · documentos: 398
Hechos verificados por el equipo (curaduría): 329
Curaduría: 415 hechos rechazados por el equipo


,subject,predicate,object,paper_id,page_start
4,INAIGEM,intervieneEn,Dirección,20.500.12748/325,3.0
19,Dirección,presentaFenomeno,Cambio climático,20.500.12748/231,4.0
44,Dirección de Investigación en Glaciares,intervieneEn,Dirección,20.500.12748/339,4.0
46,Dirección de Investigación en Ecosistemas de Montaña,intervieneEn,Dirección,20.500.12748/239,4.0
47,DIEM,intervieneEn,Dirección,20.500.12748/76,11.0
52,Dirección,ubicadaEn,Áncash,20.500.12748/604,4.0
154,Dirección,tieneEcosistema,Bofedal,20.500.12748/74,37.0
155,ANA,intervieneEn,Dirección,20.500.12748/140,31.0
173,Dirección,tieneObraSeguridad,Obras de seguridad,20.500.12748/146,3.0
221,Dirección,presentaFenomeno,Drenaje ácido de roca,20.500.12748/470,70.0


Nodos: 13,829 · relaciones: 37,264 · eventos negados quitados: 1


,Medicion,Laguna,Glaciar,Rio,Cuenca,CentroPoblado,Institucion,Nevado,Especie,Bofedal,...,Inventario,Provincia,ObraSeguridad,Departamento,MetodoMonitoreo,Fenomeno,ServicioEcosistemico,EstadoObra,Ecosistema,NivelRiesgo
nodos,2626,2038,1530,1206,1044,894,877,545,528,471,...,198,194,28,26,20,9,9,7,6,4


,MENCIONA,UBICADA_EN,TIENE_MEDICION,RELACIONADO_CON,USA_METODO,INTERVIENE_EN,PRESENTA_FENOMENO,TIENE_OBRA,MONITOREADO_CON,PUBLICADO_POR,...,INVENTARIO_DE,AFECTO_A,AMENAZA_A,OCURRIO_EN,ALIMENTADA_POR,PROVEE_SERVICIO,ESTADO_OBRA,TIENE_NIVEL_RIESGO,DRENA_HACIA,ABASTECE_A
relaciones,23359,4910,2626,1430,687,603,577,524,453,398,...,190,168,147,137,135,106,90,40,31,4


## 5. Capa 2a — Ontología OWL 🦉
Clases con jerarquía, relaciones con dominio y rango, `ubicadaEn` transitiva y clases disjuntas.
**Alineación a estándares:** lugares → `geo:Feature` (GeoSPARQL) · mediciones y métodos → `sosa:Observation` y `sosa:Procedure` (SOSA/SSN) ·
documentos, inventarios, hechos e instituciones → `prov:Entity` y `prov:Organization` (PROV-O). Se abre en **Protégé**.

In [ ]:
ONT = construir_ontologia()
ruta_owl = os.path.join(CARPETA_GRAFO, "glaciares.owl")
ONT.serialize(ruta_owl, format="pretty-xml")
r = resumen_ontologia(ONT)
print(f"Clases: {len(r['clases'])} · propiedades de objeto: {len(r['propiedades_objeto'])} · propiedades de datos: {len(r['propiedades_datos'])}")
print("Guardada:", ruta_owl)
alineacion = pd.DataFrame([{"clase": c, "es un (propio)": ", ".join(s), "estándar": ", ".join(e), "descripción": d}
                           for c, (s, e, d) in CLASES.items()])
with pd.option_context("display.max_colwidth", 80, "display.max_rows", 40):
    display(alineacion)

Clases: 30 · propiedades de objeto: 22 · propiedades de datos: 9
Guardada: /content/drive/MyDrive/proyecto_gc/grafo_glaciares/glaciares.owl


,clase,es un (propio),estándar,descripción
0,RasgoGeografico,,geo:Feature,Elemento del paisaje de montaña con ubicación (GeoSPARQL Feature).
1,MasaDeHielo,RasgoGeografico,,Hielo glaciar: glaciares y nevados.
2,Glaciar,MasaDeHielo,,Masa de hielo permanente; se inventaría por cordillera.
3,Nevado,MasaDeHielo,,Montaña con cobertura glaciar.
4,Laguna,RasgoGeografico,,"Laguna de origen glaciar (periglaciar, proglaciar o supraglaciar)."
5,Bofedal,"RasgoGeografico, Ecosistema",,Humedal altoandino con nombre propio.
6,Cordillera,RasgoGeografico,,Cordillera glaciar del Perú (20 según el INAIGEM).
7,UnidadHidrografica,RasgoGeografico,,"Cuenca, subcuenca, microcuenca o quebrada."
8,Cuenca,UnidadHidrografica,,"Cuenca, subcuenca o microcuenca."
9,Quebrada,UnidadHidrografica,,Quebrada o valle glaciar.


## 6. Capa 1 — Cargar el grafo en Neo4j ☁️

In [ ]:
NEO = None
if CONFIG["USAR_NEO4J"] and NEO4J_URI and NEO4J_PASSWORD:
    try:
        NEO = Neo4jGrafo(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
        print("Conectado a Neo4j ✅")
        if CONFIG["VACIAR_NEO4J"]:
            NEO.vaciar()
        NEO.esquema()
        NEO.cargar(N, R, CONFIG["LOTE"])
        cn, cr = NEO.conteos()
        print(f"En Neo4j: {cn.nodos.sum():,} nodos · {cr.relaciones.sum():,} relaciones")
        display(cn.head(12)); display(cr.head(12))
    except Exception as e:
        NEO = None
        print("⚠️ No se pudo usar Neo4j (se sigue sin él):", str(e)[:300])
        print("   Revisar: instancia en 'Running', URI con neo4j+s://, usuario y contraseña en Secrets.")
else:
    print("Neo4j omitido (sin claves). El resto del notebook funciona igual.")

Conectado a Neo4j ✅
Grafo vaciado (13,829 nodos borrados)
   nodos cargados · 9 s
   relaciones cargadas · 21 s
En Neo4j: 13,829 nodos · 37,264 relaciones


,tipo,nodos
0,Medicion,2626
1,Laguna,2038
2,Glaciar,1530
3,Rio,1206
4,Cuenca,1044
5,CentroPoblado,894
6,Institucion,877
7,Nevado,545
8,Especie,528
9,Bofedal,471


,relacion,relaciones
0,MENCIONA,23359
1,UBICADA_EN,4910
2,TIENE_MEDICION,2626
3,RELACIONADO_CON,1430
4,USA_METODO,687
5,INTERVIENE_EN,603
6,PRESENTA_FENOMENO,577
7,TIENE_OBRA,524
8,MONITOREADO_CON,453
9,PUBLICADO_POR,398


## 7. Capa 1 — Las 10 preguntas de competencia en Cypher 🎯
Cada respuesta trae **documento, página y frase de evidencia**. ✅ todos los saltos · 🟡 respuesta parcial · ❌ sin respuesta.

In [ ]:
if NEO:
    EVAL, RESP = evaluar(NEO.q)
    os.makedirs(os.path.join(CARPETA_GRAFO, "respuestas_cypher"), exist_ok=True)
    for k, d in RESP.items():
        d.to_csv(os.path.join(CARPETA_GRAFO, "respuestas_cypher", f"{k}.csv"), index=False, encoding="utf-8")
    with pd.option_context("display.max_colwidth", 90):
        display(EVAL)
else:
    EVAL, RESP = pd.DataFrame(), {}
    print("Sin Neo4j: ver la sección 9 (las mismas preguntas clave en SPARQL).")

,pregunta,enunciado,estado,filas,segundos,error
0,P1,¿Cuántas lagunas de origen glaciar se formaron en la Cordillera Blanca según el invent...,✅,5,0.09,
1,P2,¿Qué lagunas glaciares de Áncash tienen riesgo muy alto de desborde?,✅,5,0.09,
2,P3,¿Qué centros poblados están aguas abajo de lagunas con riesgo alto y qué estudios los ...,✅,75,0.20,
3,P4,"¿Qué lagunas tuvieron eventos de desborde, de qué glaciar provienen y qué poblados afe...",✅,83,0.14,
4,P5,¿Qué glaciares con retroceso medido (balance de masa) alimentan lagunas que hoy están ...,✅,19,0.15,
5,P6,¿Qué métodos de monitoreo se aplicaron a lagunas en riesgo y qué institución realizó c...,✅,37,0.20,
6,P7,¿En qué subcuencas coinciden lagunas en riesgo y contaminación por drenaje ácido de roca?,✅,19,0.15,
7,P8,¿Qué bofedales están en subcuencas con glaciares en retroceso y qué servicios ecosisté...,✅,5,0.24,
8,P9,¿Cómo cambió la superficie glaciar de una cordillera entre los inventarios de 2018 y 2...,✅,19,0.11,
9,P10,"¿Qué lagunas en riesgo cuentan con obras de seguridad, en qué estado están y qué pobla...",✅,66,0.13,


In [ ]:
# ver cada respuesta
for k in ["P1", "P2", "P3", "P4", "P5"]:
    if k in RESP:
        print(f"\n{k} · {CONSULTAS[k]['titulo']}")
        with pd.option_context("display.max_colwidth", 70):
            display(RESP[k].head(8))


P1 · ¿Cuántas lagunas de origen glaciar se formaron en la Cordillera Blanca según el inventario de 2023?


,inventario,dato,valor,documento,pagina,evidencia
0,Inventario Blanca 2023,registraGlaciares,511 glaciares,20.500.12748/499,68,Tabla 5 (p. 66): Blanca | glaciares libres y cubiertos: 511 (424.8...
1,Inventario Blanca 2023,registraLagunas,850 lagunas,20.500.12748/499,77,Tabla 8 (p. 75): Blanca | periglaciares 826 | proglaciares 23 | su...
2,Inventario Blanca 2023,registraLagunasNuevas,25 lagunas,20.500.12748/499,82,Figura 31 (p. 80): lagunas nuevas en Blanca: 25 (periodo 2016-2020)
3,Inventario Blanca 2023,registraLagunasProglaciares,23 lagunas,20.500.12748/499,77,Tabla 8 (p. 75): Blanca | periglaciares 826 | proglaciares 23 | su...
4,Inventario Blanca 2023,registraSuperficieGlaciar,424.86 km2,20.500.12748/499,68,Tabla 5 (p. 66): Blanca | glaciares libres y cubiertos: 511 (424.8...



P2 · ¿Qué lagunas glaciares de Áncash tienen riesgo muy alto de desborde?


,laguna,nivel,n_docs,documento,pagina,evidencia
0,Llaca,Muy alto,2,20.500.12748/668,58,Los mapas de peligro por escenario por el desborde de la laguna Ll...
1,Cuchillacocha,Muy alto,1,20.500.12748/608,68,"En Áncash, la laguna de mayor riesgo es Palcacocha, que representa..."
2,Palcacocha,Muy alto,1,20.500.12748/608,68,"En Áncash, la laguna de mayor riesgo es Palcacocha, que representa..."
3,Parón,Muy alto,1,20.500.12543/5682,39,• Se confirma el nivel de peligro muy alto en el entorno de la lag...
4,Tullparaju,Muy alto,1,20.500.12748/608,68,"En Áncash, la laguna de mayor riesgo es Palcacocha, que representa..."



P3 · ¿Qué centros poblados están aguas abajo de lagunas con riesgo alto y qué estudios los evaluaron?


,laguna,nivel,poblado,via,subcuenca,estudios,documento,pagina,evidencia
0,Palcacocha,Muy alto,Huaraz,el texto lo dice,,"[20.500.12748/676, 20.500.12748/672, 20.500.12748/668, 20.500.1274...",20.500.12543/5802,5,identificó a la laguna Palcacocha como una laguna peligrosa y aler...
1,Upiscocha,Alto,Ocongate,el texto lo dice,,"[20.500.12748/666, 20.500.12748/651, 20.500.12748/629, 20.500.1274...",20.500.12748/608,43,"Los poblados situados aguas abajo de la laguna Upiscocha, como Yan..."
2,Upiscocha,Alto,Yanama,el texto lo dice,,"[20.500.12748/667, 20.500.12748/608, 20.500.12748/605]",20.500.12748/608,43,"Los poblados situados aguas abajo de la laguna Upiscocha, como Yan..."
3,Upiscocha,Alto,Ccarhuayo,el texto lo dice,,"[20.500.12748/608, 20.500.12748/605]",20.500.12748/608,43,"Los poblados situados aguas abajo de la laguna Upiscocha, como Yan..."
4,Parón,Muy alto,Carhuaz,el texto lo dice,,"[20.500.12748/648, 20.500.12748/647, 20.500.12748/502, 20.500.1274...",20.500.12748/140,61,Se argumenta que las simulaciones de flujos de detritos en el sect...
5,Parón,Muy alto,Caraz,el texto lo dice,,"[20.500.12748/675, 20.500.12748/648, 20.500.12748/627, 20.500.1274...",20.500.12748/139,49,"Esta laguna podría afectar la estabilidad de la laguna Parón, gene..."
6,Parón,Muy alto,Cumpayhuara,el texto lo dice,,[20.500.12748/608],20.500.12748/608,52,"Los poblados situados aguas abajo de la laguna de Parón, como Cono..."
7,Tullparaju,Muy alto,Huaraz,el texto lo dice,,"[20.500.12748/672, 20.500.12748/608, 20.500.12748/516, 20.500.1274...",20.500.12748/608,49,"Aguas abajo de la laguna Tullparaju, al igual que en la laguna Pal..."



P4 · ¿Qué lagunas tuvieron eventos de desborde, de qué glaciar provienen y qué poblados afectaron?


,evento,anio,laguna,glaciar_origen,poblados_afectados,victimas,documento,pagina,evidencia
0,Deslizamiento Tambillo 1870,1870.0,Tambillo,[],"[Huaraz, Carhuaz, Macashca, Rampac Chico, Ancash]",[],20.500.12543/4673,3.0,"• 1702, Inundación de la ciudad de Huaraz • 1725, Sismo ocasiona a..."
1,Aluvión Tambillo 1883,1883.0,Tambillo,[],"[Huaraz, Carhuaz, Macashca, Rampac Chico, Ancash]",[],20.500.12543/1507,30.0,"24 de junio de 1883, a consecuencia del desborde de la laguna Tamb..."
2,Avalancha Solterococha 1917,1917.0,Solterococha,[],"[Huaraz, Yungay, Carhuaz, Huari, Ranrahirca]",[],20.500.12543/4673,3.0,"Huaraz). • 1917, Alud del Huascarán hacia el cerro de Puchgoj arra..."
3,Avalancha Palcacocha 1932,1932.0,Palcacocha,"[Palcaraju, Coropuna, Pucaranra, Ranrapalca, Randolph]",[Huaraz],"[4,000]",20.500.12543/2919,15.0,Laguna Palcacocha en el año 1932 nueve a11o antes de que experirn:...
4,Aluvión Pakliashcocha 1938,1938.0,Pakliashcocha,[],[],[],20.500.12748/502,63.0,Kinzl’s research is notable because he provided one of the first d...
5,Aluvión Arteza 1938,1938.0,Arteza,[],[Carhuaz],[],20.500.12748/627,43.0,"20/01/1938 Aluvión en la quebrada Ulta, provincia de Carhuaz, por ..."
6,Aluvión Artesa 1938,1938.0,Artesa,[],[],[],20.500.12748/628,18.0,El día 3 de marzo del 2025 en horas de la tarde una avalancha de h...
7,Aluvión Palcacocha 1941,1941.0,Palcacocha,"[Palcaraju, Coropuna, Pucaranra, Ranrapalca, Randolph]",[Huaraz],[],20.500.12748/134,97.0,"la laguna Palcacocha, que en 1941 fue origen de un aluvión catastr..."



P5 · ¿Qué glaciares con retroceso medido (balance de masa) alimentan lagunas que hoy están en riesgo?


,glaciar,retroceso,laguna,nivel,documento,pagina,evidencia
0,Ausangate,retroceso reportado,Upiscocha,Alto,20.500.12748/390,6,El paisaje predominante en el entorno de la laguna Upiscocha lo co...
1,Llaca,retroceso reportado,Llaca,Muy alto,20.500.12748/194,13,"Del retroceso del glaciar Llaca nace la laguna Llaca, la cual aún ..."
2,Palcaraju,retroceso reportado,Pacliashcocha,Alto,20.500.12543/2004,2,al pie de una lengua glaciar procedente de los nevados Palcaraju
3,Pucaranra,retroceso reportado,Palcacocha,Muy alto,20.500.12748/305,3,El glaciar Pucaranra (Ver fotografía N°03) situado en la cara oest...
4,Ranrapalca,retroceso reportado,Palcacocha,Muy alto,20.500.12543/4718,8,"Se ubica en la quebrada Cojup, muy cerca a la laguna Palcacocha; s..."
5,Salkantay,retroceso reportado,Salkantaycocha,Alto,20.500.12748/371,3,El domingo 23 de febrero del 2020 aproximadamente a las 14:42 hrs....
6,Sarapo,retroceso reportado,Quesillococha,Muy alto,20.500.12748/667,25,"Los flujos de hielo, roca y detritos, en parte influenciados por e..."
7,Vallunaraju,retroceso reportado,Artizón,Alto,20.500.12748/667,74,"En Áncash, se identificaron el riesgo de flujo de detritos en el g..."


In [ ]:
for k in ["P6", "P7", "P8", "P9", "P10"]:
    if k in RESP:
        print(f"\n{k} · {CONSULTAS[k]['titulo']}")
        with pd.option_context("display.max_colwidth", 70):
            display(RESP[k].head(8))
if "P9" in RESP:
    print("\nP9 · cambio de superficie 2018 → 2023:")
    display(comparar_superficie(RESP["P9"], 2018, 2023))


P6 · ¿Qué métodos de monitoreo se aplicaron a lagunas en riesgo y qué institución realizó cada estudio?


,laguna,metodo,instituciones_en_texto,institucion_del_documento,documento,pagina,evidencia
0,Artizón,Batimetría,"[INAIGEM, Unidad de Glaciología]",[INAIGEM],20.500.12748/253,58,De acuerdo al último levantamiento batimétrico realizado en la lag...
1,Artizón,Fotogrametría,"[INAIGEM, Unidad de Glaciología]",[INAIGEM],20.500.12748/667,23,"Para las lagunas Artizon Alto y Artizon Bajo, se tiene un modelo d..."
2,Cuchillacocha,Batimetría,"[ANA, GRA, Autoridad Administrativa del Agua, SAT]",[ANA],20.500.12543/5678,37,Actualizar el estudio de topografía y batimetría para comparar los...
3,Cuchillacocha,Limnígrafo / limnímetro,"[ANA, GRA, Autoridad Administrativa del Agua, SAT]",[ANA],20.500.12543/5802,11,Con el fin de salvaguardar la vida de la población Huaracina. • Se...
4,Gangrajanca,Batimetría,"[Unidad de Glaciología, ANA, INDECI, Municipalidad Distrital de Qu...",[INAIGEM],20.500.12748/648,24,"Nupe, sobre la base de los resultados obtenidos y la profundidad d..."
5,Gangrajanca,Dron (UAV),"[Unidad de Glaciología, ANA, INDECI, Municipalidad Distrital de Qu...",[INAIGEM],20.500.12748/648,23,"En ese contexto, se rea lizaron las siguientes actividades: evalua..."
6,Gangrajanca,Fotogrametría,"[Unidad de Glaciología, ANA, INDECI, Municipalidad Distrital de Qu...",[INAIGEM],20.500.12748/648,23,"En ese contexto, se rea lizaron las siguientes actividades: evalua..."
7,Gangrajanca,Modelamiento de flujo (HEC-RAS / FLO-2D / RAMMS),"[Unidad de Glaciología, ANA, INDECI, Municipalidad Distrital de Qu...",[INAIGEM],20.500.12748/639,110,El modelamiento numérico del potencial desborde de la laguna Gangr...



P7 · ¿En qué subcuencas coinciden lagunas en riesgo y contaminación por drenaje ácido de roca?


,subcuenca,lagunas_en_riesgo,con_drenaje_acido,documentos,evidencias
0,Accrigua,[Palcacocha],[Palcacocha],[20.500.12748/140],"[Para ello, bajo la lógica de una cadena de procesos, se simula un..."
1,Casca,[Llaca],"[Instituto, Subdirección, Llaca]","[20.500.12748/662, 20.500.12748/667, 20.500.12748/664]","[En la actualidad, es una de las zonas de estudio de la cordillera..."
2,Casca-Llaca,[Llaca],[Llaca],[20.500.12748/664],"[La inspección realizada el martes 2 de octubre, se desarrolló a l..."
3,Cojup,"[Palcacocha, Tullparaju, Cuchillacocha]",[Palcacocha],[20.500.12748/663],[Esta situación contrasta con otros sectores de la subcuenca del r...
4,Cojup valley,[Palcacocha],[Palcacocha],[20.500.12748/140],"[Para ello, bajo la lógica de una cadena de procesos, se simula un..."
5,Llaca,[Llaca],[Llaca],[20.500.12748/664],"[El único punto con pH dentro del ECA para agua, categoría 4-E2 fu..."
6,Llaca Lake basin,[Llaca],[Llaca],[20.500.12748/664],"[El único punto con pH dentro del ECA para agua, categoría 4-E2 fu..."
7,Palcacocha,[Palcacocha],[Palcacocha],[20.500.12748/140],"[Para ello, bajo la lógica de una cadena de procesos, se simula un..."



P8 · ¿Qué bofedales están en subcuencas con glaciares en retroceso y qué servicios ecosistémicos brindan?


...

,anio,anio_imagenes,km2,confianza,documento,pagina,evidencia
0,1955,NaN,726.26,0.85,20.500.12543/4601,47,Tabla (p. 47): N° | Cordillera | Área glaciar 1955/1962 (km²) | Ár...
1,1970,NaN,723.37,0.80,20.500.12748/57,74,"Entre 1970 y 1989, se realiza el primer inventario de las diecioch..."
2,1975,NaN,726.26,0.80,20.500.12748/57,74,Estas cifras fueron nuevamente analizadas por el INAIGEM a través ...
3,1989,1962.0,723.37,1.00,20.500.12748/499,182,Anexo 1 (p. 180): Blanca | 1962: 723.37 km² | 2016: 449.75 km² | 2...
4,1997,NaN,611.48,0.80,20.500.12748/57,75,El material básico de trabajo fueron las imágenes del satélite LAN...
5,2006,NaN,25.00,0.75,20.500.12542/2017,3,Usando el inventario de glaciares del Perú publicado por Hidrandin...
6,2012,NaN,527.62,0.75,20.500.12543/490,20,La superficie glaciar total en la Cordillera Blanca se ha calculad...
7,2012,NaN,195.75,0.75,20.500.12543/490,33,� En las 3 últimas décadas la Cordillera Blanca ha experimentado i...



P10 · ¿Qué lagunas en riesgo cuentan con obras de seguridad, en qué estado están y qué poblados protegen?


,laguna,nivel,obra,estado,poblados_protegidos,documento,pagina,evidencia
0,Artizón,Alto,Canal de desagüe,[Colapsada],[],20.500.12748/253,9,"Al respecto, cabe destacar que, los nevados Pucahirca y Ranrahirca..."
1,Artizón,Alto,Dique,[Colapsada],[],20.500.12748/253,67,"Es de precisar que, el dique natural pre existente de la laguna Ar..."
2,Artizón,Alto,Obras de seguridad,"[Colapsada, Deteriorada]",[],20.500.12748/253,9,"Al respecto, cabe destacar que, los nevados Pucahirca y Ranrahirca..."
3,Artizón,Alto,Presa,[Colapsada],[],20.500.12543/4673,3,"Huaraz). • 1917, Alud del Huascarán hacia el cerro de Puchgoj arra..."
4,Chaclan,Muy alto,Dique,[],[],20.500.12748/608,79,"Entre las lagunas con riesgo de desborde en Huánuco, se han identi..."
5,Cuchillacocha,Muy alto,Dique,[Deteriorada],"[Huaraz, Huancayo]",20.500.12748/608,28,"A partir de 1951, se dio inicio a la ejecución de las primeras obr..."
6,Cuchillacocha,Muy alto,Obras de seguridad,[Deteriorada],"[Huaraz, Huancayo]",20.500.12748/608,28,"A partir de 1951, se dio inicio a la ejecución de las primeras obr..."
7,Cuchillacocha,Muy alto,Presa,[],"[Huaraz, Huancayo]",20.500.12748/139,132,• Realizar la construcción de una presa de seguridad que contenga ...



P9 · cambio de superficie 2018 → 2023:


,anio_1,km2_1,anio_2,km2_2,cambio_km2,cambio_%,fuente_1,fuente_2,nota
0,2018,449.75,2023,424.86,-24.89,-5.5,20.500.12748/499 p.182,20.500.12748/499 p.68,


## 8. Capa 1 — Preguntas que el grafo responde muy bien (negación, conteo, ranking, camino) 💪
Preguntas **nuevas**, distintas de las 10: así se prepara la respuesta a preguntas imprevistas del jurado.

In [ ]:
if NEO:
    EVAL_X, RESP_X = evaluar(NEO.q, EXTRAS)
    for k, d in RESP_X.items():
        d.to_csv(os.path.join(CARPETA_GRAFO, "respuestas_cypher", f"{k}.csv"), index=False, encoding="utf-8")
    with pd.option_context("display.max_colwidth", 90):
        display(EVAL_X)
    for k in ["E1_negacion_sin_obras", "E3_conteo_departamento", "E10_camino"]:
        print(f"\n{EXTRAS[k]['titulo']}")
        display(RESP_X[k].head(8))
    if not len(RESP_X.get("E10_camino", [])):
        print("Camino calculado en Python (respaldo):")
        display(camino_local(N, R, "Huascarán", "Yungay"))
else:
    EVAL_X, RESP_X = pd.DataFrame(), {}
    print("Sin Neo4j · camino Huascarán → Yungay calculado en Python:")
    display(camino_local(N, R, "Huascarán", "Yungay"))

,pregunta,enunciado,estado,filas,segundos,error
0,E1_negacion_sin_obras,Negación · ¿Qué lagunas en riesgo (alto o muy alto) NO tienen obras de seguridad regis...,✅,6,0.10,
1,E2_negacion_sin_monitoreo,Negación · ¿Qué lagunas en riesgo no tienen ningún método de monitoreo registrado?,✅,10,0.08,
2,E3_conteo_departamento,Conteo · ¿Cuántas lagunas en riesgo hay por departamento?,✅,7,0.08,
3,E4_conteo_cordillera,Conteo con varios saltos · ¿Qué cordillera tiene más lagunas en riesgo?,✅,13,0.09,
4,E5_ranking_volumen,Ranking · Las 5 lagunas de Áncash con mayor volumen,✅,5,0.10,
5,E6_linea_tiempo,Línea de tiempo · Eventos registrados antes de 1980,✅,71,0.11,
6,E7_instituciones,Conteo · Instituciones que intervienen en más lagunas,✅,10,0.09,
7,E8_ficha,Ficha · Todo lo que se sabe de la laguna Palcacocha (con fuente),✅,40,0.09,
8,E9_comencion,Co-mención · Documentos que hablan a la vez de Palcacocha y Huaraz,✅,65,0.10,
9,E10_camino,Camino · ¿Cómo se conecta el nevado Huascarán con Yungay? (sin pasar por documentos),✅,3,0.10,



Negación · ¿Qué lagunas en riesgo (alto o muy alto) NO tienen obras de seguridad registradas?


,laguna,nivel,departamentos,documento,pagina,evidencia
0,A-10,Alto,[Cusco],20.500.12748/667,33,"En octubre de 2020, el INAIGEM reportó el crec..."
1,A-40,Alto,None,20.500.12748/406,6, Realizar la inspección a las lagunas catalog...
2,A10,Alto,None,20.500.12748/390,5,"Además, se recomendó realizar la inspección a ..."
3,A40,Alto,None,20.500.12748/390,5,"Además, se recomendó realizar la inspección a ..."
4,Ananta,Alto,[Puno],20.500.12748/667,32,Estos peligros podrían desencadenar un desbord...
5,Ananta Cucho,Alto,[Puno],20.500.12748/667,32,Estos peligros podrían desencadenar un desbord...



Conteo · ¿Cuántas lagunas en riesgo hay por departamento?


,departamento,lagunas_en_riesgo,lagunas
0,Áncash,9,"[Palcacocha, Parón, Artizón, Tullparaju, Llaca..."
1,Huánuco,3,"[Gangrajanca, Quesillococha, Chaclan]"
2,Cusco,2,"[Upiscocha, A-10]"
3,Puno,2,"[Ananta, Ananta Cucho]"
4,Apurímac,1,[Salkantaycocha]
5,Lima,1,[Pacliashcocha]
6,Pasco,1,[Lecma]



Camino · ¿Cómo se conecta el nevado Huascarán con Yungay? (sin pasar por documentos)


,camino,relaciones,documentos
0,"[Huascarán, Áncash, Yungay]","[ubicadaEn, ubicadaEn]","[20.500.12748/641, 20.500.12543/4718]"
1,"[Huascarán, Áncash, Yungay]","[ubicadaEn, ubicadaEn]","[20.500.12748/641, 20.500.12748/627]"
2,"[Huascarán, Yungay]",[ubicadaEn],[20.500.12748/641]


## 9. Capa 2b — Grafo RDF y SPARQL: deducción, procedencia y validación 🧠
La ontología **trabaja**: `ubicadaEn` es transitiva (si Palcacocha está en Quillcay y Quillcay en Áncash → Palcacocha está en Áncash),
las clases tienen jerarquía (una Laguna es un `geo:Feature`) y cada hecho es un `eb:Hecho` con `prov:wasDerivedFrom` (documento y página).

In [ ]:
RDF = construir_rdf(N, R, ONT)
ruta_ttl = os.path.join(CARPETA_GRAFO, "grafo_glaciares.ttl.gz")
with gzip.open(ruta_ttl, "wb") as f:
    f.write(RDF.serialize(format="turtle").encode("utf-8"))
print("Guardado:", ruta_ttl, f"({os.path.getsize(ruta_ttl) / 1e6:.1f} MB)")

RDF: 183,189 triples · 12,791 hechos con procedencia · 4 s
Guardado: /content/drive/MyDrive/proyecto_gc/grafo_glaciares/grafo_glaciares.ttl.gz (2.3 MB)


In [ ]:
RESP_S = {}
for k, c in SPARQL.items():
    try:
        RESP_S[k] = sparql(RDF, c["sparql"])
    except Exception as e:          # una consulta con problema no detiene las demás
        RESP_S[k] = pd.DataFrame()
        print(f"⚠️ {k}: {str(e)[:200]}")
    print(f"\n{k} · {c['titulo']}  →  {len(RESP_S[k])} filas")
    with pd.option_context("display.max_colwidth", 70):
        display(RESP_S[k].head(8))


S1_P2_deduccion · P2 con deducción · Lagunas con riesgo muy alto en Áncash (el departamento puede deducirse subiendo por ubicadaEn+)  →  5 filas


,laguna,como,documento,pagina,evidencia
0,Cuchillacocha,dicho en el texto,20.500.12748/608,68,"En Áncash, la laguna de mayor riesgo es Palcacocha, que representa..."
1,Llaca,dicho en el texto,20.500.12748/668,58,Los mapas de peligro por escenario por el desborde de la laguna Ll...
2,Palcacocha,dicho en el texto,20.500.12748/608,68,"En Áncash, la laguna de mayor riesgo es Palcacocha, que representa..."
3,Parón,dicho en el texto,20.500.12543/5682,39,• Se confirma el nivel de peligro muy alto en el entorno de la lag...
4,Tullparaju,dicho en el texto,20.500.12748/608,68,"En Áncash, la laguna de mayor riesgo es Palcacocha, que representa..."



S2_deduccion_departamentos · Deducción · ¿Cuántas lagunas tienen departamento dicho en el texto y cuántas solo deducido (ubicadaEn es transitiva)?  →  2 filas


,como,lagunas
0,dicho en el texto,199
1,solo deducido,323



S3_jerarquia · Jerarquía de clases · Elementos geográficos por clase (rdf:type/rdfs:subClassOf* geo:Feature)  →  12 filas


...

,laguna



S6_P4_eventos · P4 · Eventos en lagunas, glaciar de origen y poblados afectados  →  83 filas


,evento,anio,laguna,glaciar_origen,afecto_a
0,Desborde Platococha 2025,NaN,Platococha,,
1,Avalancha Salkantaycocha 2020,NaN,Salkantaycocha,Salkantay,
2,Deslizamiento ArtizónAlto 2012,NaN,Artizon Alto,"Taulliraju, Vallunaraju",
3,Desborde Pacliashcocha 1977,NaN,Pacliashcocha,"Palcaraju, Pucaranra, Tocllaraju",
4,Aluvión Pacliashcocha 1997,NaN,Pacliashcocha,"Palcaraju, Pucaranra, Tocllaraju",
5,Desborde Palcacocha 1941,NaN,Palcacocha,"Palcaraju, Pucaranra, Ranrapalca, Randolph, Coropuna",
6,Aluvión Palcacocha,NaN,Palcacocha,"Palcaraju, Pucaranra, Ranrapalca, Randolph, Coropuna",
7,Avalancha Pacliash,NaN,Pacliash,"Tocllaraju, Tacllaraju, Palcaraju, TocHarraju",



S7_P9_superficie · P9 · Superficie glaciar de la Cordillera Blanca por inventario (SOSA: Medicion = observación)  →  19 filas


,inventario,anio,km2,documento,pagina
0,Inventario Blanca 1955,1955,726.26,20.500.12543/4601,47
1,Inventario Blanca 1970,1970,723.37,20.500.12748/57,74
2,Inventario Blanca 1975,1975,726.26,20.500.12748/57,74
3,Inventario Blanca 1989,1989,723.37,20.500.12748/499,182
4,Inventario Blanca 1997,1997,611.48,20.500.12748/57,75
5,Inventario Blanca 2006,2006,25.00,20.500.12542/2017,3
6,Inventario Blanca 2012,2012,527.62,20.500.12543/490,20
7,Inventario Blanca 2012,2012,195.75,20.500.12543/490,33


In [ ]:
print("Validación con las reglas de la ontología (0 = sin errores):")
VALID = validar_rdf(RDF)
display(VALID)
if VALID.violaciones.sum():
    print("Detalle de lo que la ontología marcó (para revisar y, si es error, pasarlo a hechos_rechazados.csv):")
    det = cons[(cons.predicate == "tieneNivelRiesgo") & (cons.subject_type != "Laguna")]
    with pd.option_context("display.max_colwidth", 120):
        display(det[["subject", "subject_type", "object", "paper_id", "page_start", "evidencia"]])
# consistencia entre capas: P2 en Cypher y en SPARQL deben coincidir
if NEO and "P2" in RESP:
    a, b = set(RESP["P2"].laguna), set(RESP_S["S1_P2_deduccion"].laguna)
    print(f"P2 · Cypher: {len(a)} lagunas · SPARQL: {len(b)} · en común: {len(a & b)}")
    print("   solo SPARQL (deducidas por la jerarquía):", sorted(b - a)[:10])

Validación con las reglas de la ontología (0 = sin errores):


,regla,violaciones
0,amenazaA hacia algo que no es unidad territorial,0
1,ubicadaEn hacia algo que no es un lugar (geo:F...,0
2,nivel de riesgo en algo que no es laguna,0
3,individuos en clases disjuntas (rasgo geográfi...,0
4,hechos sin documento de origen,0


P2 · Cypher: 5 lagunas · SPARQL: 5 · en común: 5
   solo SPARQL (deducidas por la jerarquía): []


## 10. Capa 3 — Vistas interactivas 🗺️
Se abren en el navegador, funcionan sin internet. Pasar el mouse por una flecha muestra la fuente y la evidencia.

In [ ]:
from IPython.display import HTML, display as _display
n1, e1 = subgrafo(N, R, CONFIG["ENTIDAD_DEMO"], saltos=1, max_nodos=60)
v1 = vista_interactiva(n1, e1, os.path.join(CARPETA_GRAFO, f"vista_{CONFIG['ENTIDAD_DEMO']}.html"),
                       f"Vecindario de {CONFIG['ENTIDAD_DEMO']}", CONFIG["ENTIDAD_DEMO"])
n2, e2 = vista_riesgo(N, R, CONFIG["DEPARTAMENTO_DEMO"], ("Muy alto",))
v2 = vista_interactiva(n2, e2, os.path.join(CARPETA_GRAFO, f"vista_riesgo_{CONFIG['DEPARTAMENTO_DEMO']}.html"),
                       f"Lagunas con riesgo muy alto en {CONFIG['DEPARTAMENTO_DEMO']}: poblados, obras y glaciares")
print("Guardadas:", v1, "\n           ", v2)
_display(HTML(open(v2, encoding="utf-8").read()))

Guardadas: /content/drive/MyDrive/proyecto_gc/grafo_glaciares/vista_Palcacocha.html 
            /content/drive/MyDrive/proyecto_gc/grafo_glaciares/vista_riesgo_Áncash.html


In [ ]:
_display(HTML(open(v1, encoding="utf-8").read()))

## 11. Exportar consultas, CSV y reporte

In [ ]:
archivo_cypher(os.path.join(CARPETA_GRAFO, "consultas.cypher"))
archivo_sparql(os.path.join(CARPETA_GRAFO, "consultas.sparql"))
exportar_csv_neo4j(N, R, CARPETA_GRAFO)
with open(os.path.join(CARPETA_GRAFO, "README_grafo.md"), "w", encoding="utf-8") as f:
    f.write(README_GRAFO)
lin = ["# Reporte del grafo (semana 7)", "", f"Versión {VERSION}", "",
       f"- Nodos: {len(N):,} · relaciones: {len(R):,}",
       f"- Ontología: {len(r['clases'])} clases · {len(r['propiedades_objeto'])} propiedades de objeto · {len(r['propiedades_datos'])} de datos",
       f"- RDF: {len(RDF):,} triples", ""]
if len(EVAL):
    lin += ["## 10 preguntas (Cypher)", "", EVAL.drop(columns=["error"]).to_markdown(index=False), ""]
if len(EVAL_X):
    lin += ["## Preguntas extra (Cypher)", "", EVAL_X.drop(columns=["error"]).to_markdown(index=False), ""]
lin += ["## Validación (SPARQL)", "", VALID.to_markdown(index=False)]
with open(os.path.join(CARPETA_GRAFO, "reporte_grafo.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(lin))
print(sorted(os.listdir(CARPETA_GRAFO)))

['README_grafo.md', 'consultas.cypher', 'consultas.sparql', 'glaciares.owl', 'grafo_glaciares.ttl.gz', 'nodos.csv', 'relaciones.csv', 'reporte_grafo.md', 'respuestas_cypher', 'vista_Palcacocha.html', 'vista_riesgo_Áncash.html']


## 12. Para la demo en Neo4j (Aura → *Query*)
```cypher
// Vecindario de Palcacocha (clic en un nodo para expandir)
MATCH p = (l:Laguna {k: 'palcacocha'})-[r]-(x) WHERE NOT type(r) IN ['MENCIONA'] RETURN p LIMIT 60

// Lagunas con riesgo muy alto en Áncash y a quién amenazan
MATCH p = (l:Laguna)-[:TIENE_NIVEL_RIESGO]->(:NivelRiesgo {nombre: 'Muy alto'})
WHERE 'Áncash' IN l.departamentos
OPTIONAL MATCH q = (l)-[:AMENAZA_A]->(:CentroPoblado)
RETURN p, q
```
Todas las consultas están en `consultas.cypher` (se pueden pegar en Neo4j Browser).

## Entrega (GitHub: `data/grafo/` y `notebooks/`)
Este notebook, `glaciares.owl`, `consultas.cypher`, `consultas.sparql`, `reporte_grafo.md`, `README_grafo.md`,
las vistas `vista_*.html` y la carpeta `respuestas_cypher/`. `grafo_glaciares.ttl.gz`, `nodos.csv` y `relaciones.csv` si pesan menos de 25 MB.